##  GEMINI OUTPUT, TABLES OUTPUT IN JSON AND MMD AS WELL, ANALYSIS FROM GEMINI ON ERRORS , CER , CONVERTED PDF OF GD TO JSON USING GEMINI SINCE LATEX OF GROUND WAS NOT RIGHT


# send the ground truth to gemini

In [2]:
import os
import json
import time
from datetime import datetime
from dotenv import load_dotenv
import google.generativeai as genai
import sys
from pathlib import Path

# Handle both notebook and script environments
try:
    # Try to get the script directory (works in .py files)
    script_dir = Path(__file__).parent
except NameError:
    # Fallback for Jupyter notebooks
    script_dir = Path.cwd()
    print("⚠️  Running in notebook environment, using current working directory")

# Add the path to access prompt_store.py using relative path
project_root = script_dir.parent.parent.parent
ocr_path = project_root / "ocr"
sys.path.append(str(ocr_path))

try:
    from prompt_store import v15
    print("✅ Successfully imported v15 prompt")
except ImportError as e:
    print(f"❌ Failed to import prompt_store: {e}")
    print(f"🔍 Tried to import from: {ocr_path}")
    print(f"📁 Current script directory: {script_dir}")
    print(f"📁 Project root: {project_root}")
    sys.exit(1)

# === Load API Key ===
load_dotenv()

model_name = "gemini-2.5-pro"
api_key = os.getenv("GOOGLE_GEMINI_API")

if not api_key:
    print("❌ GOOGLE_GEMINI_API environment variable not found!")
    print("Please make sure you have a .env file with your API key")
    sys.exit(1)

genai.configure(api_key=api_key)
model = genai.GenerativeModel(model_name)

class ProcessingTracker:
    def __init__(self):
        self.total_files = 0
        self.processed_files = 0
        self.successful_files = 0
        self.failed_files = 0
        self.json_files = 0
        self.text_files = 0
        self.errors = []
        self.start_time = None
        self.end_time = None
    
    def start_processing(self):
        self.start_time = datetime.now()
        print(f"🚀 Started processing at {self.start_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print("=" * 60)
    
    def end_processing(self):
        self.end_time = datetime.now()
        duration = self.end_time - self.start_time
        print("\n" + "=" * 60)
        print("📊 PROCESSING SUMMARY")
        print("=" * 60)
        print(f"Total PDF files found:     {self.total_files}")
        print(f"Successfully processed:    {self.successful_files}")
        print(f"Failed to process:         {self.failed_files}")
        print(f"Valid JSON outputs:        {self.json_files}")
        print(f"Text outputs (invalid JSON): {self.text_files}")
        print(f"Processing time:           {duration}")
        print(f"Completed at:              {self.end_time.strftime('%Y-%m-%d %H:%M:%S')}")
        
        if self.errors:
            print(f"\n❌ ERRORS ENCOUNTERED ({len(self.errors)}):")
            print("-" * 40)
            for i, error in enumerate(self.errors, 1):
                print(f"{i}. {error}")
        else:
            print(f"\n✅ No errors encountered!")
        print("=" * 60)
    
    def add_error(self, error_msg):
        self.errors.append(error_msg)
        self.failed_files += 1

def send_pdf_to_gemini_and_save_json(pdf_path, prompt, output_base_dir, tracker, file_index):
    try:
        pdf_filename = os.path.basename(pdf_path)
        print(f"\n📄 [{file_index}/{tracker.total_files}] Processing: {pdf_filename}")
        
        # Upload the PDF file
        print("   ⬆️  Uploading PDF to Gemini...")
        file_resource = genai.upload_file(pdf_path, mime_type="application/pdf")
        
        # Compose the prompt and file
        print("   🤖 Generating content with Gemini...")
        start_time = time.time()
        response = model.generate_content([prompt, file_resource])
        end_time = time.time()
        generated_text = response.text
        
        processing_time = end_time - start_time
        print(f"   ⏱️  Gemini processing time: {processing_time:.2f} seconds")

        # Extract the filename and set output directory for JSON file
        pdf_stem = os.path.splitext(pdf_filename)[0]
        output_dir = os.path.join(output_base_dir, pdf_stem)
        os.makedirs(output_dir, exist_ok=True)

        # Try to validate JSON before saving
        try:
            # Attempt to parse as JSON to validate
            json_data = json.loads(generated_text)
            output_json_path = os.path.join(output_dir, f"{pdf_stem}.json")
            # Save as properly formatted JSON
            with open(output_json_path, 'w', encoding='utf-8') as output_file:
                json.dump(json_data, output_file, indent=2, ensure_ascii=False)
            print(f"   ✅ Valid JSON saved: {output_json_path}")
            tracker.json_files += 1
            tracker.successful_files += 1
            
        except json.JSONDecodeError as json_error:
            # If not valid JSON, save as text file
            output_txt_path = os.path.join(output_dir, f"{pdf_stem}.json")
            with open(output_txt_path, 'w', encoding='utf-8') as output_file:
                output_file.write(generated_text)
            print(f"   ⚠️  Invalid JSON, saved as text: {output_txt_path}")
            print(f"   📝 JSON Error: {str(json_error)[:100]}...")
            tracker.text_files += 1
            tracker.successful_files += 1
        
        tracker.processed_files += 1
        
    except Exception as e:
        error_msg = f"File: {pdf_filename} - Error: {str(e)}"
        print(f"   ❌ Error processing {pdf_filename}: {str(e)}")
        tracker.add_error(error_msg)

# Function to process all PDF files in a directory
def process_all_pdfs(input_dir, output_dir, prompt):
    tracker = ProcessingTracker()
    
    if not os.path.exists(input_dir):
        print(f"❌ Input directory does not exist: {input_dir}")
        print(f"🔍 Tried path: {input_dir}")
        return tracker
    
    # First, count all PDF files
    print(f"🔍 Scanning for PDF files in: {input_dir}")
    pdf_files = []
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".pdf"):
                pdf_files.append(os.path.join(root, file))
    
    tracker.total_files = len(pdf_files)
    print(f"📁 Found {tracker.total_files} PDF files")
    
    if tracker.total_files == 0:
        print("❌ No PDF files found in the specified directory")
        return tracker
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    print(f"📂 Output directory: {output_dir}")
    
    tracker.start_processing()
    
    # Process each PDF file
    for index, pdf_path in enumerate(pdf_files, 1):
        send_pdf_to_gemini_and_save_json(pdf_path, prompt, output_dir, tracker, index)
        
        # Show progress
        progress = (index / tracker.total_files) * 100
        print(f"   📈 Progress: {progress:.1f}% ({index}/{tracker.total_files})")
    
    tracker.end_processing()
    return tracker

# === Main Function for Easy Usage ===
def main(input_dir=None, output_dir=None):
    print("🎯 PDF to JSON Processor with Gemini AI")
    print("=" * 60)
    
    # Use provided paths or default relative paths
    if input_dir is None:
        input_pdf_dir = script_dir / "Chemistry_human" / "Chemistry_pdf_docx_human_ocr"
    else:
        input_pdf_dir = Path(input_dir)
    
    if output_dir is None:
        output_json_dir = script_dir / "Chemistry_human" / "pdf_docx_json"
    else:
        output_json_dir = Path(output_dir)
    
    # Convert to strings for compatibility
    input_pdf_dir = str(input_pdf_dir)
    output_json_dir = str(output_json_dir)
    
    print(f"📂 Input directory:  {input_pdf_dir}")
    print(f"📂 Output directory: {output_json_dir}")
    print(f"🤖 Using model:      {model_name}")
    print(f"📋 Using prompt:     v15")
    
    # Start processing all PDFs in the input directory
    result_tracker = process_all_pdfs(input_pdf_dir, output_json_dir, v15)
    
    # Final status
    if result_tracker.total_files > 0:
        success_rate = (result_tracker.successful_files / result_tracker.total_files) * 100
        print(f"\n🎉 Overall success rate: {success_rate:.1f}%")
        
        if result_tracker.failed_files > 0:
            print(f"⚠️  {result_tracker.failed_files} files failed to process")
        else:
            print("🎊 All files processed successfully!")
    else:
        print("❌ No files were processed")
    
    return result_tracker

# === Example Usage ===
if __name__ == "__main__":
    main()

⚠️  Running in notebook environment, using current working directory
✅ Successfully imported v15 prompt
🎯 PDF to JSON Processor with Gemini AI
📂 Input directory:  /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/Chemistry_pdf_docx_human_ocr
📂 Output directory: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/pdf_docx_json
🤖 Using model:      gemini-2.5-pro
📋 Using prompt:     v15
🔍 Scanning for PDF files in: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/Chemistry_pdf_docx_human_ocr
📁 Found 2 PDF files
📂 Output directory: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/pdf_docx_json
🚀 Started processing at 2025-07-30 18:52:48

📄 [1/2] Processing: 01

## to clean the json 

In [3]:
import os
import json

def clean_json_content(file_path):
    """
    Cleans the JSON file by removing the first and last lines from the file.
    """
    try:
        # Check if the file is empty
        if os.path.getsize(file_path) == 0:
            print(f"Skipping empty file: {file_path}")
            return

        # Open the file to read the raw content
        with open(file_path, 'r') as file:
            lines = file.readlines()

        # Ensure the file has more than two lines (i.e., has content to remove from both ends)
        if len(lines) <= 2:
            print(f"Skipping file with not enough content to clean: {file_path}")
            return
        
        # Remove the first and last lines
        cleaned_lines = lines[1:-1]

        # Join the cleaned lines and load the cleaned data as JSON
        cleaned_data = "".join(cleaned_lines)
        try:
            data = json.loads(cleaned_data)
        except json.JSONDecodeError as e:
            print(f"Error reading JSON from file {file_path}: {e}")
            return

        # Save the cleaned JSON data back to the file
        with open(file_path, 'w') as file:
            json.dump(data, file, indent=2)
        
        print(f"Successfully cleaned and saved: {file_path}")
    
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")

def clean_json_files_in_directory(directory_path):
    """
    Loops through the directory and all its subdirectories, cleaning each JSON file.
    """
    for subdir, _, files in os.walk(directory_path):
        for filename in files:
            if filename.endswith('.json'):
                file_path = os.path.join(subdir, filename)
                clean_json_content(file_path)

# Set the directory path where the JSON files are located
directory_path = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/pdf_docx_json"

# Clean all JSON files in the directory and its subdirectories
clean_json_files_in_directory(directory_path)


Successfully cleaned and saved: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/pdf_docx_json/02_10021039271083421101694954824/02_10021039271083421101694954824.json
Successfully cleaned and saved: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/pdf_docx_json/01_10021008481039611101693742355/01_10021008481039611101693742355.json


## to break the json into each question for each chapter for ground truth

In [5]:
import os
import json

def create_solution_jsons(input_file_path, output_directory):
    """
    Create individual solution JSON files for each question from the JSON files.
    """
    try:
        # Read the input JSON file
        with open(input_file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)
        
        # Extract the folder name from the file path
        # Get the parent directory name (e.g., C01_1234567890)
        folder_path = os.path.dirname(input_file_path)
        folder_name = os.path.basename(folder_path)
        
        # Extract the prefix (first part before first underscore)
        folder_prefix = folder_name.split('_')[0] if '_' in folder_name else folder_name
        
        # Create output folder structure
        output_folder = os.path.join(output_directory, folder_name)
        os.makedirs(output_folder, exist_ok=True)
        
        print(f"Processing folder: {folder_name}")
        print(f"Output directory: {output_folder}")
        
        # Loop through each question and create a new JSON for each
        for entry in data:
            question_number = entry.get('question_number')
            solution_text = entry.get('solution_text', '')  # Taking solution_text for each solution
            diagrams = entry.get('diagrams', [])
            pages = entry.get('pages', [])
            
            # Create the solution structure with solution_text as the solution content
            solution = [{
                "question_number": question_number,
                "solution_text": solution_text,  # This contains the solution text
                "diagrams": diagrams,
                "pages": pages
            }]
            
            # Define the solution file path with the new naming convention
            solution_file_path = os.path.join(output_folder, f"{folder_prefix}_solution_{question_number}.json")
            
            # Write the solution JSON to the file
            with open(solution_file_path, 'w', encoding='utf-8') as solution_file:
                json.dump(solution, solution_file, indent=2, ensure_ascii=False)
            
            print(f"  ✅ Solution {question_number} saved: {solution_file_path}")
    
    except Exception as e:
        print(f"❌ Error processing file {input_file_path}: {e}")

def process_output_directory(input_directory, output_directory):
    """
    Process all JSON files in the specified directory structure.
    """
    processed_count = 0
    error_count = 0
    
    print(f"🔍 Scanning directory: {input_directory}")
    print(f"📂 Output will be saved to: {output_directory}")
    print("=" * 60)
    
    # Walk through all subdirectories
    for root, dirs, files in os.walk(input_directory):
        for filename in files:
            if filename.endswith('.json'):
                file_path = os.path.join(root, filename)
                
                try:
                    create_solution_jsons(file_path, output_directory)
                    processed_count += 1
                except Exception as e:
                    print(f"❌ Error processing {file_path}: {e}")
                    error_count += 1
    
    print("=" * 60)
    print(f"📊 PROCESSING SUMMARY:")
    print(f"✅ Successfully processed: {processed_count} files")
    print(f"❌ Errors encountered: {error_count} files")
    print("=" * 60)

def main():
    """
    Main function to run the solution creation process.
    """
    print("🎯 Chemistry Solution JSON Creator")
    print("=" * 60)
    
    # Input directory containing the JSON files - CORRECTED PATH
    input_directory = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/pdf_docx_json"
    
    # Output directory where solutions will be saved - CORRECTED PATH
    output_directory = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/solution_chapters"
    
    # Verify input directory exists
    if not os.path.exists(input_directory):
        print(f"❌ Input directory does not exist: {input_directory}")
        return
    
    # Create output directory if it doesn't exist
    os.makedirs(output_directory, exist_ok=True)
    
    print(f"📂 Input directory:  {input_directory}")
    print(f"📂 Output directory: {output_directory}")
    
    # Process all files
    process_output_directory(input_directory, output_directory)

if __name__ == "__main__":
    main()

🎯 Chemistry Solution JSON Creator
📂 Input directory:  /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/pdf_docx_json
📂 Output directory: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/solution_chapters
🔍 Scanning directory: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/pdf_docx_json
📂 Output will be saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/solution_chapters
Processing folder: 02_10021039271083421101694954824
Output directory: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/solution_chapters/02_10021039271083421101694954824
  ✅ Solution 1 saved: 

## send the PREDICTION IMAGES TO GEMINI PRO

In [ ]:
# 🚀 CLAUDE WITH GEMINI PREPROCESSING - BEST OF BOTH WORLDS (CORRECTED)
import sys, os, time, json, shutil, pandas as pd
from dotenv import load_dotenv
import glob
import hashlib
import base64
import requests
from typing import List, Optional, Dict, Any
import subprocess
import re

print("🚀 CLAUDE SONNET 4 WITH GEMINI PREPROCESSING - CORRECTED VERSION!")
print("=" * 70)

# === SETUP ===
physics_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_claude_pcmb/maths"
parent_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading"
solution_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement"

# CORRECTED CSV PATH
hw_solution_with_qb_meta_csv = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_claude_pcmb/hw_df_with_solutions_and_questions.csv"

# PDF DIRECTORY TO PROCESS
PDF_DIRECTORY = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_claude_pcmb/maths/maths_Gemini/maths"

# Claude Vertex AI Configuration
CLAUDE_CONFIG = {
    "endpoint": "us-east5-aiplatform.googleapis.com",
    "location_id": "us-east5", 
    "project_id": "llm-sandbox-426711",
    "model_id": "claude-sonnet-4",
    "method": "rawPredict"
}

load_dotenv(os.path.join(solution_dir, ".env"))
sys.path.insert(0, physics_dir)
sys.path.insert(1, parent_dir)

class MockST:
    def __init__(self): self.secrets = {'GOOGLE_GEMINI_API': os.getenv('GOOGLE_GEMINI_API', '')}
    def error(self, m): print(f'❌ {m}')
    def info(self, m): print(f'ℹ️  {m}')
    def warning(self, m): print(f'⚠️  {m}')
    def success(self, m): print(f'✅ {m}')
sys.modules['streamlit'] = MockST()

# === IMPORT GEMINI PREPROCESSING FUNCTIONS ===
try:
    # Import ALL the Gemini processing functions you love!
    from processors import ImageProcessor  # Your existing ImageProcessor
    from config import create_pdf_output_structure, IMAGE_RESIZE_DIM  # Your config
    from cache_utils import create_pdf_request_hash, load_cached_response, save_cached_response  # Your caching
    from results_manager import ResultsManager  # Your results manager
    print("✅ All Gemini preprocessing functions imported successfully!")
except ImportError as e:
    print(f"❌ Failed to import Gemini functions: {e}")
    print("Please ensure processors.py, config.py, etc. are available")

# === SETUP PROMPT STORE ===
print("📝 Loading prompt store...")
ocr_dir = os.path.join(parent_dir, 'ocr')
sys.path.insert(0, ocr_dir)

try:
    import prompt_store as ps
    print("✅ Prompt store imported successfully")
    if hasattr(ps, 'v13'):
        print("✅ v13 prompt found in prompt store")
    else:
        print("❌ v13 prompt not found in prompt store")
        sys.exit(1)
except ImportError as e:
    print(f"❌ CRITICAL: Failed to import prompt store from {ocr_dir}")
    sys.exit(1)

# === CLAUDE API FUNCTIONS (ONLY THING THAT CHANGES) ===

def get_access_token():
    """Get Google Cloud access token using gcloud CLI"""
    try:
        result = subprocess.run(
            ['gcloud', 'auth', 'print-access-token'],
            capture_output=True, text=True, check=True
        )
        return result.stdout.strip()
    except subprocess.CalledProcessError as e:
        print(f"❌ Failed to get access token: {e}")
        return None

def convert_pil_image_to_base64(pil_image, max_size_mb=1.5):
    """Convert PIL Image to base64 (for Claude) with compression"""
    try:
        from PIL import Image
        import io
        
        # Convert to RGB if necessary
        if pil_image.mode in ('RGBA', 'LA', 'P'):
            pil_image = pil_image.convert('RGB')
        
        max_size_bytes = max_size_mb * 1024 * 1024
        
        # Try different quality levels until under size limit
        for quality in [85, 70, 60, 50, 40]:
            buffer = io.BytesIO()
            pil_image.save(buffer, format='JPEG', quality=quality, optimize=True)
            
            if buffer.tell() <= max_size_bytes:
                print(f"🔧 Compressed PIL image to quality {quality} ({buffer.tell()/1024/1024:.1f}MB)")
                return base64.b64encode(buffer.getvalue()).decode('utf-8')
        
        # If still too large, resize
        print("🔧 Resizing PIL image...")
        width, height = pil_image.size
        scale_factor = 0.7  # Reduce by 30%
        new_width = int(width * scale_factor)
        new_height = int(height * scale_factor)
        
        resized_image = pil_image.resize((new_width, new_height), Image.Resampling.LANCZOS)
        buffer = io.BytesIO()
        resized_image.save(buffer, format='JPEG', quality=70, optimize=True)
        
        print(f"✅ Resized to {new_width}x{new_height} ({buffer.tell()/1024/1024:.1f}MB)")
        return base64.b64encode(buffer.getvalue()).decode('utf-8')
        
    except Exception as e:
        print(f"❌ Failed to convert PIL image: {e}")
        return None

def create_claude_content_from_gemini_format(content_items):
    """Convert Gemini-style content (with PIL Images) to Claude format"""
    claude_content = []
    
    for item in content_items:
        if isinstance(item, str):
            # Text content - same as Gemini
            claude_content.append({
                "type": "text",
                "text": item
            })
        else:
            # Check if it's a PIL Image (Gemini format)
            try:
                from PIL import Image
                if isinstance(item, Image.Image):
                    # Convert PIL Image to base64 for Claude
                    base64_image = convert_pil_image_to_base64(item)
                    if base64_image:
                        claude_content.append({
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": "image/jpeg",
                                "data": base64_image
                            }
                        })
            except ImportError:
                pass
    
    return claude_content

def robust_json_parser(raw_text):
    """
    Robust JSON parser with multiple fallback strategies
    """
    cleaned = raw_text.strip('```json').strip('```').strip()
    
    # Strategy 1: Direct JSON parsing
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"⚠️  Direct JSON parsing failed: {e}")
    
    # Strategy 2: Fix common JSON issues
    try:
        # Fix common formatting issues
        fixed_json = cleaned
        
        # Remove any BOM or weird characters at the start
        if fixed_json.startswith('\ufeff'):
            fixed_json = fixed_json[1:]
        
        # Fix unescaped quotes in strings
        lines = fixed_json.split('\n')
        fixed_lines = []
        
        for line in lines:
            # If line has uneven quotes, try to fix
            quote_count = line.count('"')
            if quote_count % 2 != 0 and not line.strip().endswith(','):
                # Try to add missing quote at the end of the line
                line = line.rstrip() + '"'
            fixed_lines.append(line)
        
        fixed_json = '\n'.join(fixed_lines)
        
        # Try parsing the fixed JSON
        return json.loads(fixed_json)
    except json.JSONDecodeError as e2:
        print(f"⚠️  JSON fix attempt failed: {e2}")
    
    # Strategy 3: Extract JSON from response if it's embedded
    try:
        # Look for JSON array or object patterns
        json_pattern = r'(\[.*\]|\{.*\})'
        matches = re.search(json_pattern, cleaned, re.DOTALL)
        if matches:
            potential_json = matches.group(1)
            return json.loads(potential_json)
        else:
            print("❌ No JSON pattern found in response")
    except json.JSONDecodeError as e3:
        print(f"⚠️  JSON extraction failed: {e3}")
    
    # Strategy 4: Try to repair truncated JSON
    try:
        # If JSON is truncated, try to close it
        fixed_json = cleaned
        
        # Handle truncated string values
        # If the JSON ends with incomplete string, close it properly
        if not fixed_json.rstrip().endswith((']', '}', '"')):
            # Find the last opening quote
            last_quote_pos = fixed_json.rfind('"')
            if last_quote_pos != -1:
                # Check if this quote is opening a string value (not closing)
                quotes_before = fixed_json[:last_quote_pos].count('"')
                if quotes_before % 2 == 1:  # Odd number means we have an opening quote
                    fixed_json += '"'  # Close the string
        
        # Count open braces/brackets
        open_braces = fixed_json.count('{') - fixed_json.count('}')
        open_brackets = fixed_json.count('[') - fixed_json.count(']')
        
        # Add missing closing characters
        fixed_json += '}' * open_braces
        fixed_json += ']' * open_brackets
        
        parsed = json.loads(fixed_json)
        print(f"✅ Successfully repaired truncated JSON - recovered {len(parsed)} questions")
        return parsed
    except json.JSONDecodeError as e4:
        print(f"⚠️  JSON repair failed: {e4}")
        
    # Strategy 5: Extract valid portion of JSON array
    try:
        # Try to extract complete questions from partial JSON
        print("🔧 Attempting to extract valid questions from truncated response...")
        
        # Find the last complete question object
        import re
        
        # Split by question objects and take only complete ones
        question_pattern = r'\{\s*"question_number":\s*\d+,.*?\}'
        questions = re.findall(question_pattern, cleaned, re.DOTALL)
        
        if questions:
            # Reconstruct valid JSON array
            valid_json = '[' + ','.join(questions) + ']'
            parsed = json.loads(valid_json)
            print(f"✅ Extracted {len(parsed)} complete questions from truncated response")
            return parsed
        else:
            print("❌ Could not extract any complete questions")
    except json.JSONDecodeError as e5:
        print(f"⚠️  Question extraction failed: {e5}")
    except Exception as e6:
        print(f"⚠️  Unexpected error in question extraction: {e6}")
    
    print(f"❌ All JSON parsing strategies failed")
    print(f"📄 FULL RAW RESPONSE (for debugging):")
    print("=" * 80)
    print(raw_text)
    print("=" * 80)
    return None

def send_to_claude_with_gemini_preprocessing(content: List, cache_dir: Optional[str] = None,
                                           pdf_name: str = None, prompt_version: str = None, 
                                           image_size: int = None, questions_count: int = None) -> Optional[dict]:
    """
    Send Gemini-preprocessed content to Claude API
    This replaces send_to_gemini_with_cache() but keeps the same interface!
    """
    
    # Use same caching logic as Gemini
    model_name = "claude-sonnet-4"
    request_hash = create_pdf_request_hash(
        pdf_file_path=pdf_name,
        prompt_text=content[0] if content else "",
        prompt_version=prompt_version,
        image_dimension=image_size or IMAGE_RESIZE_DIM,
        model_name=model_name,
        questions=None
    )
    
    # Check cache (same as Gemini)
    if cache_dir:
        cached_response = load_cached_response(cache_dir, request_hash)
        if cached_response:
            print(f"✅ Using cached response (saved {cached_response.get('processing_time_seconds', 0):.2f}s)")
            return cached_response.get('response_data')
    
    # Get access token
    access_token = get_access_token()
    if not access_token:
        print("❌ Failed to get access token")
        return None
    
    # Convert Gemini-style content to Claude format
    claude_content = create_claude_content_from_gemini_format(content)
    
    # Check total size
    total_size = sum(len(json.dumps(item).encode()) for item in claude_content) / (1024 * 1024)
    print(f"📊 Request size: {total_size:.1f}MB")
    
    if total_size > 25:
        print("⚠️  Request approaching 30MB limit!")
    
    # Prepare Claude request
    request_payload = {
        "anthropic_version": "vertex-2023-10-16",
        "stream": False,
        "max_tokens": 20000,  # ← Increased to handle longer responses
        "temperature": 0.1,
        "top_p": 0.95,
        "top_k": 40,
        "messages": [
            {
                "role": "user",
                "content": claude_content
            }
        ]
    }
    
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json; charset=utf-8"
    }
    
    url = f"https://{CLAUDE_CONFIG['endpoint']}/v1/projects/{CLAUDE_CONFIG['project_id']}/locations/{CLAUDE_CONFIG['location_id']}/publishers/anthropic/models/{CLAUDE_CONFIG['model_id']}:{CLAUDE_CONFIG['method']}"
    
    # Make API call (same timing as Gemini)
    start_time = time.time()
    try:
        print(f"🤖 Making API call to Claude Sonnet 4...")
        response = requests.post(url, headers=headers, json=request_payload, timeout=180)
        processing_time = time.time() - start_time
        
        if response.status_code == 200:
            response_data = response.json()
            
            if 'content' in response_data and len(response_data['content']) > 0:
                raw_text = response_data['content'][0]['text']
                
                # Use robust JSON parser - FIXED!
                parsed = robust_json_parser(raw_text)
                
                if parsed:
                    print(f"✅ API call completed in {processing_time:.2f}s")
                    
                    # Save to cache (same as Gemini)
                    if cache_dir:
                        save_cached_response(
                            cache_dir=cache_dir,
                            request_hash=request_hash,
                            response_data=parsed,
                            processing_time=processing_time,
                            pdf_name=pdf_name,
                            prompt_version=prompt_version,
                            image_size=image_size,
                            questions_count=questions_count
                        )
                    
                    return parsed
                else:
                    return None
            else:
                print("❌ No content in Claude response")
                return None
        else:
            print(f"❌ Claude API error ({response.status_code}): {response.text}")
            return None
            
    except Exception as e:
        processing_time = time.time() - start_time
        print(f"❌ Failed to process content with Claude after {processing_time:.2f}s: {e}")
        return None

# === USE ALL YOUR EXISTING GEMINI FUNCTIONS ===

# Import HTML cleaner (same as before)
try:
    from html_text_cleaner import clean_html_to_text as extract_text_from_html
    print("✅ HTML cleaner imported successfully")
except ImportError as e:
    def extract_text_from_html(html_text):
        import re
        if not html_text:
            return html_text
        clean = re.sub(r'<[^>]+>', '', html_text)
        clean = re.sub(r'\s+', ' ', clean).strip()
        return clean

# Keep ALL your existing helper functions exactly as they are
def remove_prefix(filename):
    """Remove numeric prefix like 01_, 02_, etc. from filename"""
    if '_' in filename:
        parts = filename.split('_', 1)
        if parts[0].isdigit():
            return parts[1]
    return filename

def extract_pdf_name_from_url(url):
    """Extract PDF name from UPLOADED_ANS URL"""
    if pd.isna(url) or not isinstance(url, str):
        return None
    filename = url.split('/')[-1]
    if filename.endswith('.pdf'):
        filename = filename[:-4]
    return filename

def clean_escaped_html(text):
    """Clean both regular HTML and escaped HTML tags"""
    if not text:
        return text
    text = text.replace('\\/', '/')
    cleaned = extract_text_from_html(text)
    return cleaned

def extract_question_from_json_content(json_content):
    """Extract and clean question text from JSON content"""
    try:
        data = json.loads(json_content)
        if isinstance(data, list) and len(data) > 0:
            question_data = data[0]
        else:
            question_data = data
        
        question_text = None
        if 'questionStem' in question_data and 'text' in question_data['questionStem']:
            question_text = question_data['questionStem']['text']
        
        if question_text:
            if question_text.startswith('"') and question_text.endswith('"'):
                question_text = question_text[1:-1]
            cleaned_text = clean_escaped_html(question_text)
            return cleaned_text
        
        return None
    except Exception as e:
        print(f"❌ Error extracting question: {e}")
        return None

def load_questions_for_pdf_with_json_parsing(pdf_name, csv_path):
    """Load questions from CSV, parse JSON content, and clean HTML"""
    try:
        if not hasattr(load_questions_for_pdf_with_json_parsing, 'cached_df'):
            print(f"📊 Loading CSV: {csv_path}")
            load_questions_for_pdf_with_json_parsing.cached_df = pd.read_csv(csv_path, low_memory=False)
            print(f"✅ CSV loaded: {len(load_questions_for_pdf_with_json_parsing.cached_df)} rows")
            
            load_questions_for_pdf_with_json_parsing.cached_df['pdf_name_extracted'] = \
                load_questions_for_pdf_with_json_parsing.cached_df['UPLOADED_ANS'].apply(extract_pdf_name_from_url)
        
        df = load_questions_for_pdf_with_json_parsing.cached_df
        base_pdf_name = remove_prefix(pdf_name.replace('.pdf', ''))
        pdf_df = df[df['pdf_name_extracted'].str.contains(base_pdf_name, na=False)]
        
        if len(pdf_df) == 0:
            print(f"⚠️  No questions found for PDF: {base_pdf_name}")
            return []
        
        print(f"✅ Found {len(pdf_df)} rows for PDF: {base_pdf_name}")
        
        questions = []
        for idx, row in pdf_df.iterrows():
            raw_content = None
            if pd.notna(row.get('content')):
                raw_content = str(row['content'])
            elif pd.notna(row.get('textsolutions')):
                raw_content = str(row['textsolutions'])
            
            if raw_content:
                cleaned_question = extract_question_from_json_content(raw_content)
                if cleaned_question and cleaned_question.strip():
                    questions.append(cleaned_question)
        
        print(f"📝 Extracted {len(questions)} cleaned questions for {pdf_name}")
        return questions
        
    except Exception as e:
        print(f"❌ Error loading questions for {pdf_name}: {e}")
        return []

def get_model_name():
    return "claude-sonnet-4"

# === CLAUDE OCR FUNCTIONS (USING GEMINI PREPROCESSING) ===

def ocr_pdf_claude_with_gemini_preprocessing(pdf_file_path: str, output_folder: str, cache_dir: Optional[str], 
                                           prompt_version: str) -> Optional[dict]:
    """
    EXACT same interface as Gemini ocr_pdf, but sends to Claude at the end!
    """
    # Get prompt (same as Gemini)
    prompt = getattr(ps, prompt_version)
    model_name = get_model_name()
    pdf_name = os.path.splitext(os.path.basename(pdf_file_path))[0]
    
    # Create output structure (same as Gemini)
    pdf_output_paths = create_pdf_output_structure(pdf_name)
    output_folder = pdf_output_paths['images']
    cache_dir = pdf_output_paths['cache']
    
    # Initialize processors (SAME as Gemini - your existing ImageProcessor!)
    image_processor = ImageProcessor(output_folder)
    results_manager = ResultsManager(pdf_output_paths['json'])
    
    # Process PDF to images (SAME as Gemini - returns PIL Images!)
    page_images = image_processor.process_pdf(pdf_file_path)
    
    # Build content list (SAME as Gemini format!)
    content = [prompt]
    if page_images:
        content.extend(page_images)  # PIL Images, same as Gemini!
    
    # Send to Claude with Gemini preprocessing (ONLY this line changes!)
    print(f"🤖 Processing with Claude Sonnet 4 using Gemini preprocessing (prompt: {prompt_version})...")
    results = send_to_claude_with_gemini_preprocessing(
        content=content,
        cache_dir=cache_dir,
        pdf_name=pdf_name,
        prompt_version=prompt_version,
        image_size=IMAGE_RESIZE_DIM
    )
    
    # Save results (same as Gemini)
    if results:
        results_manager.save_pdf_results(results)
        return results
    else:
        print("❌ No results from Claude OCR")
        return None

def ocr_with_questions_claude_with_gemini_preprocessing(questions: List[str], pdf_file_path: str, output_folder: str, 
                                                      cache_dir: Optional[str], prompt_version: str) -> Optional[dict]:
    """
    EXACT same interface as Gemini ocr_with_questions, but sends to Claude!
    """
    # Get prompt (same as Gemini)
    prompt = getattr(ps, prompt_version)
    pdf_name = os.path.splitext(os.path.basename(pdf_file_path))[0]
    
    # Create output structure (same as Gemini)
    pdf_output_paths = create_pdf_output_structure(pdf_name)
    output_folder = pdf_output_paths['images']
    cache_dir = pdf_output_paths['cache']
    
    # Initialize processors (SAME as Gemini!)
    image_processor = ImageProcessor(output_folder)
    results_manager = ResultsManager(pdf_output_paths['json'])
    
    # Process images (SAME as Gemini - returns PIL Images!)
    hand_written_solutions = image_processor.process_pdf(pdf_file_path=pdf_file_path)
    
    print(f"🤖 Processing {len(questions)} questions with {len(hand_written_solutions)} solution images")
    
    # Build content list (SAME format as Gemini!)
    content = [prompt]
    if questions:
        content.extend(["# Question set : "] + questions + [" # Hand written solutions : "])
    if hand_written_solutions:
        content.extend(hand_written_solutions)  # PIL Images, same as Gemini!
    
    # Send to Claude with Gemini preprocessing (ONLY this changes!)
    print(f"🤖 Processing with Claude Sonnet 4 using Gemini preprocessing (prompt: {prompt_version})...")
    results = send_to_claude_with_gemini_preprocessing(
        content=content,
        cache_dir=cache_dir,
        pdf_name=pdf_name,
        prompt_version=prompt_version,
        image_size=IMAGE_RESIZE_DIM,
        questions_count=len(questions)
    )
    
    # Save results (same as Gemini)
    if results:
        results_manager.save_question_results(results)
        return results
    else:
        print("❌ No results from Claude OCR")
        return None

# === MAIN PROCESSING FUNCTIONS (ALMOST IDENTICAL TO GEMINI) ===

def process_single_pdf_with_questions(pdf_path, pdf_name):
    """Process a single PDF - SAME interface as Gemini version"""
    
    print(f"\n{'='*60}")
    print(f"🔧 Processing PDF: {pdf_name}")
    print(f"📄 Full path: {pdf_path}")
    
    # Load questions (same as Gemini)
    questions = load_questions_for_pdf_with_json_parsing(pdf_name, hw_solution_with_qb_meta_csv)
    
    if not questions:
        print("⚠️  No questions found, falling back to basic OCR")
        result = ocr_pdf_claude_with_gemini_preprocessing(pdf_path, physics_dir, None, "v13")
    else:
        print(f"🎯 Processing with {len(questions)} cleaned questions")
        print(f"📝 Question preview: {questions[0][:80]}..." if questions else "")
            
        # Use Claude with Gemini preprocessing
        result = ocr_with_questions_claude_with_gemini_preprocessing(
            questions=questions,
            pdf_file_path=pdf_path,
            output_folder=physics_dir,
            cache_dir=None,
            prompt_version="v13"
        )
    
    # Rest is IDENTICAL to Gemini processing...
    if result:
        pdf_base_name = pdf_name.replace('.pdf', '')
        for search_pattern in [
            f"{physics_dir}/output/{pdf_base_name}/json/output.json",
            f"{physics_dir}/output/output.json",
            f"{physics_dir}/output/OUTPUT_JSON/output.json"
        ]:
            if os.path.exists(search_pattern):
                target_dir = f"{physics_dir}/output/batch_results/{pdf_base_name}"
                os.makedirs(target_dir, exist_ok=True)
                target_path = f"{target_dir}/output.json"
                
                shutil.copy2(search_pattern, target_path)
                print(f"📄 Output saved to: {target_path}")
                
                with open(target_path, 'r') as f:
                    data = json.load(f)
                    print(f"✅ Success: {len(data)} questions processed for {pdf_name}")
                    
                    clean_count = 0
                    for item in data:
                        if 'question_text' in item:
                            if not any(tag in item['question_text'] for tag in ['</', '<div', '<strong', '<\/div', '<\/strong']):
                                clean_count += 1
                    
                    print(f"🧹 {clean_count}/{len(data)} questions are fully cleaned")
                    
                return True
        
        print(f"⚠️  Output file not found for {pdf_name}")
        return False
    else:
        print(f"❌ OCR processing failed for {pdf_name}")
        return False

def process_pdf_directory(pdf_directory):
    """Process all PDF files - SAME as Gemini version"""
    
    print(f"\n🎯 STARTING BATCH PDF PROCESSING WITH CLAUDE + GEMINI PREPROCESSING...")
    print(f"📁 Directory: {pdf_directory}")
    
    pdf_pattern = os.path.join(pdf_directory, "*.pdf")
    pdf_files = glob.glob(pdf_pattern)
    
    if not pdf_files:
        print(f"❌ No PDF files found in {pdf_directory}")
        return
    
    print(f"📋 Found {len(pdf_files)} PDF files to process")
    
    successful = 0
    failed = 0
    
    for i, pdf_path in enumerate(pdf_files, 1):
        pdf_name = os.path.basename(pdf_path)
        print(f"\n🔄 Processing {i}/{len(pdf_files)}: {pdf_name}")
        
        try:
            success = process_single_pdf_with_questions(pdf_path, pdf_name)
            if success:
                successful += 1
                print(f"✅ Successfully processed {pdf_name}")
            else:
                failed += 1
                print(f"❌ Failed to process {pdf_name}")
        except Exception as e:
            failed += 1
            print(f"❌ Error processing {pdf_name}: {e}")
    
    print(f"\n{'='*60}")
    print(f"🏁 BATCH PROCESSING COMPLETE WITH CLAUDE + GEMINI PREPROCESSING!")
    print(f"📊 Results:")
    print(f"   ✅ Successful: {successful}")
    print(f"   ❌ Failed: {failed}")
    print(f"   📁 Total files: {len(pdf_files)}")
    print(f"   📍 Results saved in: {physics_dir}/output/batch_results/")

# === EXECUTION ===
if __name__ == "__main__":
    print(f"📊 CSV: {os.path.basename(hw_solution_with_qb_meta_csv)}")
    print(f"🤖 Model: Claude Sonnet 4 with Gemini Preprocessing")
    print(f"🔗 Endpoint: {CLAUDE_CONFIG['endpoint']}")
    
    # Verify authentication
    token = get_access_token()
    if token:
        print("✅ Google Cloud authentication successful")
    else:
        print("❌ Google Cloud authentication failed")
        exit(1)
    
    # Process entire PDF directory
    process_pdf_directory(PDF_DIRECTORY)
    
    print(f"\n🏁 ALL PROCESSING COMPLETE WITH CLAUDE + GEMINI PREPROCESSING!")


🚀 CLAUDE SONNET 4 WITH GEMINI PREPROCESSING - CORRECTED VERSION!
❌ Failed to import Gemini functions: No module named 'cache_utils'
Please ensure processors.py, config.py, etc. are available
📝 Loading prompt store...
✅ Prompt store imported successfully
✅ v13 prompt found in prompt store
✅ HTML cleaner imported successfully
📊 CSV: hw_df_with_solutions_and_questions.csv
🤖 Model: Claude Sonnet 4 with Gemini Preprocessing
🔗 Endpoint: us-east5-aiplatform.googleapis.com
✅ Google Cloud authentication successful

🎯 STARTING BATCH PDF PROCESSING WITH CLAUDE + GEMINI PREPROCESSING...
📁 Directory: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_claude_pcmb/maths/maths_Gemini/maths
📋 Found 10 PDF files to process

🔄 Processing 1/10: 11_10021019941080491171694789012.pdf

🔧 Processing PDF: 11_10021019941080491171694789012.pdf
📄 Full path: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z

## to break the json into each quesiton 

In [ ]:
# 🚀 CLAUDE WITH GEMINI PREPROCESSING - BEST OF BOTH WORLDS
import sys, os, time, json, shutil, pandas as pd
from dotenv import load_dotenv
import glob
import hashlib
import base64
import requests
from typing import List, Optional, Dict, Any
import subprocess
import re

print("🚀 CLAUDE SONNET 4 WITH GEMINI PREPROCESSING!")
print("=" * 70)

# === SETUP ===
physics_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_claude_pcmb/maths"
parent_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading"
solution_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement"

# CORRECTED CSV PATH
hw_solution_with_qb_meta_csv = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_claude_pcmb/hw_df_with_solutions_and_questions.csv"

# PDF DIRECTORY TO PROCESS
PDF_DIRECTORY = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_claude_pcmb/maths/maths_Gemini/maths"

# Claude Vertex AI Configuration
CLAUDE_CONFIG = {
    "endpoint": "us-east5-aiplatform.googleapis.com",
    "location_id": "us-east5", 
    "project_id": "llm-sandbox-426711",
    "model_id": "claude-sonnet-4",
    "method": "rawPredict"
}

load_dotenv(os.path.join(solution_dir, ".env"))
sys.path.insert(0, physics_dir)
sys.path.insert(1, parent_dir)

class MockST:
    def __init__(self): self.secrets = {'GOOGLE_GEMINI_API': os.getenv('GOOGLE_GEMINI_API', '')}
    def error(self, m): print(f'❌ {m}')
    def info(self, m): print(f'ℹ️  {m}')
    def warning(self, m): print(f'⚠️  {m}')
    def success(self, m): print(f'✅ {m}')
sys.modules['streamlit'] = MockST()

# === IMPORT GEMINI PREPROCESSING FUNCTIONS ===
try:
    # Import ALL the Gemini processing functions you love!
    from processors import ImageProcessor  # Your existing ImageProcessor
    from config import create_pdf_output_structure, IMAGE_RESIZE_DIM  # Your config
    from cache_utils import create_pdf_request_hash, load_cached_response, save_cached_response  # Your caching
    from results_manager import ResultsManager  # Your results manager
    print("✅ All Gemini preprocessing functions imported successfully!")
except ImportError as e:
    print(f"❌ Failed to import Gemini functions: {e}")
    print("Please ensure processors.py, config.py, etc. are available")

# === SETUP PROMPT STORE ===
print("📝 Loading prompt store...")
ocr_dir = os.path.join(parent_dir, 'ocr')
sys.path.insert(0, ocr_dir)

try:
    import prompt_store as ps
    print("✅ Prompt store imported successfully")
    if hasattr(ps, 'v13'):
        print("✅ v13 prompt found in prompt store")
    else:
        print("❌ v13 prompt not found in prompt store")
        sys.exit(1)
except ImportError as e:
    print(f"❌ CRITICAL: Failed to import prompt store from {ocr_dir}")
    sys.exit(1)

# === CLAUDE API FUNCTIONS (ONLY THING THAT CHANGES) ===

def get_access_token():
    """Get Google Cloud access token using gcloud CLI"""
    try:
        result = subprocess.run(
            ['gcloud', 'auth', 'print-access-token'],
            capture_output=True, text=True, check=True
        )
        return result.stdout.strip()
    except subprocess.CalledProcessError as e:
        print(f"❌ Failed to get access token: {e}")
        return None

def convert_pil_image_to_base64(pil_image, max_size_mb=1.5):
    """Convert PIL Image to base64 (for Claude) with compression"""
    try:
        from PIL import Image
        import io
        
        # Convert to RGB if necessary
        if pil_image.mode in ('RGBA', 'LA', 'P'):
            pil_image = pil_image.convert('RGB')
        
        max_size_bytes = max_size_mb * 1024 * 1024
        
        # Try different quality levels until under size limit
        for quality in [85, 70, 60, 50, 40]:
            buffer = io.BytesIO()
            pil_image.save(buffer, format='JPEG', quality=quality, optimize=True)
            
            if buffer.tell() <= max_size_bytes:
                print(f"🔧 Compressed PIL image to quality {quality} ({buffer.tell()/1024/1024:.1f}MB)")
                return base64.b64encode(buffer.getvalue()).decode('utf-8')
        
        # If still too large, resize
        print("🔧 Resizing PIL image...")
        width, height = pil_image.size
        scale_factor = 0.7  # Reduce by 30%
        new_width = int(width * scale_factor)
        new_height = int(height * scale_factor)
        
        resized_image = pil_image.resize((new_width, new_height), Image.Resampling.LANCZOS)
        buffer = io.BytesIO()
        resized_image.save(buffer, format='JPEG', quality=70, optimize=True)
        
        print(f"✅ Resized to {new_width}x{new_height} ({buffer.tell()/1024/1024:.1f}MB)")
        return base64.b64encode(buffer.getvalue()).decode('utf-8')
        
    except Exception as e:
        print(f"❌ Failed to convert PIL image: {e}")
        return None

def create_claude_content_from_gemini_format(content_items):
    """Convert Gemini-style content (with PIL Images) to Claude format"""
    claude_content = []
    
    for item in content_items:
        if isinstance(item, str):
            # Text content - same as Gemini
            claude_content.append({
                "type": "text",
                "text": item
            })
        else:
            # Check if it's a PIL Image (Gemini format)
            try:
                from PIL import Image
                if isinstance(item, Image.Image):
                    # Convert PIL Image to base64 for Claude
                    base64_image = convert_pil_image_to_base64(item)
                    if base64_image:
                        claude_content.append({
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": "image/jpeg",
                                "data": base64_image
                            }
                        })
            except ImportError:
                pass
    
    return claude_content

def send_to_claude_with_gemini_preprocessing(content: List, cache_dir: Optional[str] = None,
                                           pdf_name: str = None, prompt_version: str = None, 
                                           image_size: int = None, questions_count: int = None) -> Optional[dict]:
    """
    Send Gemini-preprocessed content to Claude API
    This replaces send_to_gemini_with_cache() but keeps the same interface!
    """
    
    # Use same caching logic as Gemini
    model_name = "claude-sonnet-4"
    request_hash = create_pdf_request_hash(
        pdf_file_path=pdf_name,
        prompt_text=content[0] if content else "",
        prompt_version=prompt_version,
        image_dimension=image_size or IMAGE_RESIZE_DIM,
        model_name=model_name,
        questions=None
    )
    
    # Check cache (same as Gemini)
    if cache_dir:
        cached_response = load_cached_response(cache_dir, request_hash)
        if cached_response:
            print(f"✅ Using cached response (saved {cached_response.get('processing_time_seconds', 0):.2f}s)")
            return cached_response.get('response_data')
    
    # Get access token
    access_token = get_access_token()
    if not access_token:
        print("❌ Failed to get access token")
        return None
    
    # Convert Gemini-style content to Claude format
    claude_content = create_claude_content_from_gemini_format(content)
    
    # Check total size
    total_size = sum(len(json.dumps(item).encode()) for item in claude_content) / (1024 * 1024)
    print(f"📊 Request size: {total_size:.1f}MB")
    
    if total_size > 25:
        print("⚠️  Request approaching 30MB limit!")
    
    # Prepare Claude request
    request_payload = {
        "anthropic_version": "vertex-2023-10-16",
        "stream": False,
        "max_tokens": 4096,
        "temperature": 0.1,
        "top_p": 0.95,
        "top_k": 40,
        "messages": [
            {
                "role": "user",
                "content": claude_content
            }
        ]
    }
    
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json; charset=utf-8"
    }
    
    url = f"https://{CLAUDE_CONFIG['endpoint']}/v1/projects/{CLAUDE_CONFIG['project_id']}/locations/{CLAUDE_CONFIG['location_id']}/publishers/anthropic/models/{CLAUDE_CONFIG['model_id']}:{CLAUDE_CONFIG['method']}"
    
    # Make API call (same timing as Gemini)
    start_time = time.time()
    try:
        print(f"🤖 Making API call to Claude Sonnet 4...")
        response = requests.post(url, headers=headers, json=request_payload, timeout=180)
        processing_time = time.time() - start_time
        
        if response.status_code == 200:
            response_data = response.json()
            
            if 'content' in response_data and len(response_data['content']) > 0:
                raw_text = response_data['content'][0]['text']
                
                # Parse JSON (same as Gemini logic)
                cleaned = raw_text.strip('```json').strip('```').strip()
                try:
                    parsed = json.loads(cleaned)
                    print(f"✅ API call completed in {processing_time:.2f}s")
                    
                    # Save to cache (same as Gemini)
                    if cache_dir:
                        save_cached_response(
                            cache_dir=cache_dir,
                            request_hash=request_hash,
                            response_data=parsed,
                            processing_time=processing_time,
                            pdf_name=pdf_name,
                            prompt_version=prompt_version,
                            image_size=image_size,
                            questions_count=questions_count
                        )
                    
                    return parsed
                except json.JSONDecodeError as e:
                    print(f"❌ JSON parsing error: {e}")
                    return None
            else:
                print("❌ No content in Claude response")
                return None
        else:
            print(f"❌ Claude API error ({response.status_code}): {response.text}")
            return None
            
    except Exception as e:
        processing_time = time.time() - start_time
        print(f"❌ Failed to process content with Claude after {processing_time:.2f}s: {e}")
        return None

# === USE ALL YOUR EXISTING GEMINI FUNCTIONS ===

# Import HTML cleaner (same as before)
try:
    from html_text_cleaner import clean_html_to_text as extract_text_from_html
    print("✅ HTML cleaner imported successfully")
except ImportError as e:
    def extract_text_from_html(html_text):
        import re
        if not html_text:
            return html_text
        clean = re.sub(r'<[^>]+>', '', html_text)
        clean = re.sub(r'\s+', ' ', clean).strip()
        return clean

# Keep ALL your existing helper functions exactly as they are
def remove_prefix(filename):
    """Remove numeric prefix like 01_, 02_, etc. from filename"""
    if '_' in filename:
        parts = filename.split('_', 1)
        if parts[0].isdigit():
            return parts[1]
    return filename

def extract_pdf_name_from_url(url):
    """Extract PDF name from UPLOADED_ANS URL"""
    if pd.isna(url) or not isinstance(url, str):
        return None
    filename = url.split('/')[-1]
    if filename.endswith('.pdf'):
        filename = filename[:-4]
    return filename

def clean_escaped_html(text):
    """Clean both regular HTML and escaped HTML tags"""
    if not text:
        return text
    text = text.replace('\\/', '/')
    cleaned = extract_text_from_html(text)
    return cleaned

def extract_question_from_json_content(json_content):
    """Extract and clean question text from JSON content"""
    try:
        data = json.loads(json_content)
        if isinstance(data, list) and len(data) > 0:
            question_data = data[0]
        else:
            question_data = data
        
        question_text = None
        if 'questionStem' in question_data and 'text' in question_data['questionStem']:
            question_text = question_data['questionStem']['text']
        
        if question_text:
            if question_text.startswith('"') and question_text.endswith('"'):
                question_text = question_text[1:-1]
            cleaned_text = clean_escaped_html(question_text)
            return cleaned_text
        
        return None
    except Exception as e:
        print(f"❌ Error extracting question: {e}")
        return None

def load_questions_for_pdf_with_json_parsing(pdf_name, csv_path):
    """Load questions from CSV, parse JSON content, and clean HTML"""
    try:
        if not hasattr(load_questions_for_pdf_with_json_parsing, 'cached_df'):
            print(f"📊 Loading CSV: {csv_path}")
            load_questions_for_pdf_with_json_parsing.cached_df = pd.read_csv(csv_path, low_memory=False)
            print(f"✅ CSV loaded: {len(load_questions_for_pdf_with_json_parsing.cached_df)} rows")
            
            load_questions_for_pdf_with_json_parsing.cached_df['pdf_name_extracted'] = \
                load_questions_for_pdf_with_json_parsing.cached_df['UPLOADED_ANS'].apply(extract_pdf_name_from_url)
        
        df = load_questions_for_pdf_with_json_parsing.cached_df
        base_pdf_name = remove_prefix(pdf_name.replace('.pdf', ''))
        pdf_df = df[df['pdf_name_extracted'].str.contains(base_pdf_name, na=False)]
        
        if len(pdf_df) == 0:
            print(f"⚠️  No questions found for PDF: {base_pdf_name}")
            return []
        
        print(f"✅ Found {len(pdf_df)} rows for PDF: {base_pdf_name}")
        
        questions = []
        for idx, row in pdf_df.iterrows():
            raw_content = None
            if pd.notna(row.get('content')):
                raw_content = str(row['content'])
            elif pd.notna(row.get('textsolutions')):
                raw_content = str(row['textsolutions'])
            
            if raw_content:
                cleaned_question = extract_question_from_json_content(raw_content)
                if cleaned_question and cleaned_question.strip():
                    questions.append(cleaned_question)
        
        print(f"📝 Extracted {len(questions)} cleaned questions for {pdf_name}")
        return questions
        
    except Exception as e:
        print(f"❌ Error loading questions for {pdf_name}: {e}")
        return []

def get_model_name():
    return "claude-sonnet-4"

# === CLAUDE OCR FUNCTIONS (USING GEMINI PREPROCESSING) ===

def ocr_pdf_claude_with_gemini_preprocessing(pdf_file_path: str, output_folder: str, cache_dir: Optional[str], 
                                           prompt_version: str) -> Optional[dict]:
    """
    EXACT same interface as Gemini ocr_pdf, but sends to Claude at the end!
    """
    # Get prompt (same as Gemini)
    prompt = getattr(ps, prompt_version)
    model_name = get_model_name()
    pdf_name = os.path.splitext(os.path.basename(pdf_file_path))[0]
    
    # Create output structure (same as Gemini)
    pdf_output_paths = create_pdf_output_structure(pdf_name)
    output_folder = pdf_output_paths['images']
    cache_dir = pdf_output_paths['cache']
    
    # Initialize processors (SAME as Gemini - your existing ImageProcessor!)
    image_processor = ImageProcessor(output_folder)
    results_manager = ResultsManager(pdf_output_paths['json'])
    
    # Process PDF to images (SAME as Gemini - returns PIL Images!)
    page_images = image_processor.process_pdf(pdf_file_path)
    
    # Build content list (SAME as Gemini format!)
    content = [prompt]
    if page_images:
        content.extend(page_images)  # PIL Images, same as Gemini!
    
    # Send to Claude with Gemini preprocessing (ONLY this line changes!)
    print(f"🤖 Processing with Claude Sonnet 4 using Gemini preprocessing (prompt: {prompt_version})...")
    results = send_to_claude_with_gemini_preprocessing(
        content=content,
        cache_dir=cache_dir,
        pdf_name=pdf_name,
        prompt_version=prompt_version,
        image_size=IMAGE_RESIZE_DIM
    )
    
    # Save results (same as Gemini)
    if results:
        results_manager.save_pdf_results(results)
        return results
    else:
        print("❌ No results from Claude OCR")
        return None

def ocr_with_questions_claude_with_gemini_preprocessing(questions: List[str], pdf_file_path: str, output_folder: str, 
                                                      cache_dir: Optional[str], prompt_version: str) -> Optional[dict]:
    """
    EXACT same interface as Gemini ocr_with_questions, but sends to Claude!
    """
    # Get prompt (same as Gemini)
    prompt = getattr(ps, prompt_version)
    pdf_name = os.path.splitext(os.path.basename(pdf_file_path))[0]
    
    # Create output structure (same as Gemini)
    pdf_output_paths = create_pdf_output_structure(pdf_name)
    output_folder = pdf_output_paths['images']
    cache_dir = pdf_output_paths['cache']
    
    # Initialize processors (SAME as Gemini!)
    image_processor = ImageProcessor(output_folder)
    results_manager = ResultsManager(pdf_output_paths['json'])
    
    # Process images (SAME as Gemini - returns PIL Images!)
    hand_written_solutions = image_processor.process_pdf(pdf_file_path=pdf_file_path)
    
    print(f"🤖 Processing {len(questions)} questions with {len(hand_written_solutions)} solution images")
    
    # Build content list (SAME format as Gemini!)
    content = [prompt]
    if questions:
        content.extend(["# Question set : "] + questions + [" # Hand written solutions : "])
    if hand_written_solutions:
        content.extend(hand_written_solutions)  # PIL Images, same as Gemini!
    
    # Send to Claude with Gemini preprocessing (ONLY this changes!)
    print(f"🤖 Processing with Claude Sonnet 4 using Gemini preprocessing (prompt: {prompt_version})...")
    results = send_to_claude_with_gemini_preprocessing(
        content=content,
        cache_dir=cache_dir,
        pdf_name=pdf_name,
        prompt_version=prompt_version,
        image_size=IMAGE_RESIZE_DIM,
        questions_count=len(questions)
    )
    
    # Save results (same as Gemini)
    if results:
        results_manager.save_question_results(results)
        return results
    else:
        print("❌ No results from Claude OCR")
        return None

# === MAIN PROCESSING FUNCTIONS (ALMOST IDENTICAL TO GEMINI) ===

def process_single_pdf_with_questions(pdf_path, pdf_name):
    """Process a single PDF - SAME interface as Gemini version"""
    
    print(f"\n{'='*60}")
    print(f"🔧 Processing PDF: {pdf_name}")
    print(f"📄 Full path: {pdf_path}")
    
    # Load questions (same as Gemini)
    questions = load_questions_for_pdf_with_json_parsing(pdf_name, hw_solution_with_qb_meta_csv)
    
    if not questions:
        print("⚠️  No questions found, falling back to basic OCR")
        result = ocr_pdf_claude_with_gemini_preprocessing(pdf_path, physics_dir, None, "v13")
    else:
        print(f"🎯 Processing with {len(questions)} cleaned questions")
        print(f"📝 Question preview: {questions[0][:80]}..." if questions else "")
            
        # Use Claude with Gemini preprocessing
        result = ocr_with_questions_claude_with_gemini_preprocessing(
            questions=questions,
            pdf_file_path=pdf_path,
            output_folder=physics_dir,
            cache_dir=None,
            prompt_version="v13"
        )
    
    # Rest is IDENTICAL to Gemini processing...
    if result:
        pdf_base_name = pdf_name.replace('.pdf', '')
        for search_pattern in [
            f"{physics_dir}/output/{pdf_base_name}/json/output.json",
            f"{physics_dir}/output/output.json",
            f"{physics_dir}/output/OUTPUT_JSON/output.json"
        ]:
            if os.path.exists(search_pattern):
                target_dir = f"{physics_dir}/output/batch_results/{pdf_base_name}"
                os.makedirs(target_dir, exist_ok=True)
                target_path = f"{target_dir}/output.json"
                
                shutil.copy2(search_pattern, target_path)
                print(f"📄 Output saved to: {target_path}")
                
                with open(target_path, 'r') as f:
                    data = json.load(f)
                    print(f"✅ Success: {len(data)} questions processed for {pdf_name}")
                    
                    clean_count = 0
                    for item in data:
                        if 'question_text' in item:
                            if not any(tag in item['question_text'] for tag in ['</', '<div', '<strong', '<\/div', '<\/strong']):
                                clean_count += 1
                    
                    print(f"🧹 {clean_count}/{len(data)} questions are fully cleaned")
                    
                return True
        
        print(f"⚠️  Output file not found for {pdf_name}")
        return False
    else:
        print(f"❌ OCR processing failed for {pdf_name}")
        return False

def process_pdf_directory(pdf_directory):
    """Process all PDF files - SAME as Gemini version"""
    
    print(f"\n🎯 STARTING BATCH PDF PROCESSING WITH CLAUDE + GEMINI PREPROCESSING...")
    print(f"📁 Directory: {pdf_directory}")
    
    pdf_pattern = os.path.join(pdf_directory, "*.pdf")
    pdf_files = glob.glob(pdf_pattern)
    
    if not pdf_files:
        print(f"❌ No PDF files found in {pdf_directory}")
        return
    
    print(f"📋 Found {len(pdf_files)} PDF files to process")
    
    successful = 0
    failed = 0
    
    for i, pdf_path in enumerate(pdf_files, 1):
        pdf_name = os.path.basename(pdf_path)
        print(f"\n🔄 Processing {i}/{len(pdf_files)}: {pdf_name}")
        
        try:
            success = process_single_pdf_with_questions(pdf_path, pdf_name)
            if success:
                successful += 1
                print(f"✅ Successfully processed {pdf_name}")
            else:
                failed += 1
                print(f"❌ Failed to process {pdf_name}")
        except Exception as e:
            failed += 1
            print(f"❌ Error processing {pdf_name}: {e}")
    
    print(f"\n{'='*60}")
    print(f"🏁 BATCH PROCESSING COMPLETE WITH CLAUDE + GEMINI PREPROCESSING!")
    print(f"📊 Results:")
    print(f"   ✅ Successful: {successful}")
    print(f"   ❌ Failed: {failed}")
    print(f"   📁 Total files: {len(pdf_files)}")
    print(f"   📍 Results saved in: {physics_dir}/output/batch_results/")

# === EXECUTION ===
if __name__ == "__main__":
    print(f"📊 CSV: {os.path.basename(hw_solution_with_qb_meta_csv)}")
    print(f"🤖 Model: Claude Sonnet 4 with Gemini Preprocessing")
    print(f"🔗 Endpoint: {CLAUDE_CONFIG['endpoint']}")
    
    # Verify authentication
    token = get_access_token()
    if token:
        print("✅ Google Cloud authentication successful")
    else:
        print("❌ Google Cloud authentication failed")
        exit(1)
    
    # Process entire PDF directory
    process_pdf_directory(PDF_DIRECTORY)
    
    print(f"\n🏁 ALL PROCESSING COMPLETE WITH CLAUDE + GEMINI PREPROCESSING!")

🚀 CLAUDE SONNET 4 WITH GEMINI PREPROCESSING!
❌ Failed to import Gemini functions: No module named 'cache_utils'
Please ensure processors.py, config.py, etc. are available
📝 Loading prompt store...
✅ Prompt store imported successfully
✅ v13 prompt found in prompt store
✅ HTML cleaner imported successfully
📊 CSV: hw_df_with_solutions_and_questions.csv
🤖 Model: Claude Sonnet 4 with Gemini Preprocessing
🔗 Endpoint: us-east5-aiplatform.googleapis.com
✅ Google Cloud authentication successful

🎯 STARTING BATCH PDF PROCESSING WITH CLAUDE + GEMINI PREPROCESSING...
📁 Directory: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_claude_pcmb/maths/maths_Gemini/maths
📋 Found 10 PDF files to process

🔄 Processing 1/10: 11_10021019941080491171694789012.pdf

🔧 Processing PDF: 11_10021019941080491171694789012.pdf
📄 Full path: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr_claude_pcmb/ma

In [7]:
import os
import json

def create_solution_jsons(input_file_path, output_directory):
    """
    Create individual solution JSON files for each question in the input JSON.
    """
    try:
        # Read the input JSON file
        with open(input_file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)
        
        # Extract the folder name from the file path
        # Get the parent directory of the json folder (e.g., 01_10021008481039611101693742355)
        folder_path = os.path.dirname(os.path.dirname(input_file_path))
        folder_name = os.path.basename(folder_path)
        
        # Extract the first part of the directory name (before the first '_')
        folder_prefix = folder_name.split('_')[0]  # Gets the first part before '_'
        
        # Define output folder
        output_folder = os.path.join(output_directory, folder_name)
        os.makedirs(output_folder, exist_ok=True)
        
        print(f"Processing folder: {folder_name}")
        print(f"Output directory: {output_folder}")
        
        # Loop through each question and create a new JSON for each
        for entry in data:
            question_number = entry.get('question_number')
            solution = [{
                "question_number": question_number,
                "solution_text": entry.get('solution_text'),
                "diagrams": entry.get('diagrams', []),
                "pages": entry.get('pages', [])
            }]
            
            # Define the solution file path with the new naming convention
            solution_file_path = os.path.join(output_folder, f"{folder_prefix}_solution_{question_number}.json")
            
            # Write the solution JSON to the file
            with open(solution_file_path, 'w', encoding='utf-8') as solution_file:
                json.dump(solution, solution_file, indent=2, ensure_ascii=False)
            
            print(f"  ✅ Solution {question_number} saved: {solution_file_path}")
    
    except Exception as e:
        print(f"❌ Error processing file {input_file_path}: {e}")

def process_output_directory(input_directory, output_directory):
    """
    Process all output_with_questions.json files in the specified directory structure.
    """
    processed_count = 0
    error_count = 0
    
    print(f"🔍 Scanning directory: {input_directory}")
    print(f"📂 Output will be saved to: {output_directory}")
    print("=" * 60)
    
    # Walk through all subdirectories
    for root, dirs, files in os.walk(input_directory):
        # Look for the specific file pattern: */json/output_with_questions.json
        if 'output_with_questions.json' in files and root.endswith('json'):
            file_path = os.path.join(root, 'output_with_questions.json')
            print(f"📄 Found: {file_path}")
            
            try:
                create_solution_jsons(file_path, output_directory)
                processed_count += 1
            except Exception as e:
                print(f"❌ Error processing {file_path}: {e}")
                error_count += 1
    
    print("=" * 60)
    print(f"📊 PROCESSING SUMMARY:")
    print(f"✅ Successfully processed: {processed_count} files")
    print(f"❌ Errors encountered: {error_count} files")
    print("=" * 60)

def main():
    """
    Main function to run the solution creation process.
    """
    print("🎯 Chemistry Solution JSON Creator")
    print("=" * 60)
    
    # Input directory containing the output folders
    input_directory = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/output"
    
    # Output directory where solutions will be saved
    output_directory = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_Gemini/solutions"
    
    # Verify input directory exists
    if not os.path.exists(input_directory):
        print(f"❌ Input directory does not exist: {input_directory}")
        return
    
    # Create output directory if it doesn't exist
    os.makedirs(output_directory, exist_ok=True)
    
    print(f"📂 Input directory:  {input_directory}")
    print(f"📂 Output directory: {output_directory}")
    
    # Process all files
    process_output_directory(input_directory, output_directory)

if __name__ == "__main__":
    main()

🎯 Chemistry Solution JSON Creator
📂 Input directory:  /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/output
📂 Output directory: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_Gemini/solutions
🔍 Scanning directory: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/output
📂 Output will be saved to: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_Gemini/solutions
📄 Found: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/output/02_10021039271083421101694954824/json/output_with_questions.json
Processing folder: 02_10021039271083421101694954824
Output directory: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subject

# to make a json of human and gemini ocr

In [8]:
import os
import json

def compare_ocr_folders(human_ocr_path, gemini_ocr_path, solution_folder, solution_number, file_prefix):
    human_solution_path = os.path.join(
        human_ocr_path, 'chemistry/Chemistry_human/solution_chapters', solution_folder,
        f"{file_prefix}_solution_{solution_number}.json"
    )
    gemini_solution_path = os.path.join(
        gemini_ocr_path, 'chemistry/Chemistry_Gemini/solutions', solution_folder,
        f"{file_prefix}_solution_{solution_number}.json"
    )
    
    print(f"Processing: {human_solution_path}")
    print(f"Processing: {gemini_solution_path}")
    
    # Initialize simplified data structure (only text and question number)
    combined_data = {
        "question_number": solution_number,
        "human_text": "NA",
        "gemini_text": "NA"
    }
    
    # Human OCR
    if os.path.exists(human_solution_path):
        with open(human_solution_path, 'r') as human_file:
            try:
                human_data = json.load(human_file)
                if isinstance(human_data, list) and len(human_data) > 0:
                    human_item = human_data[0]
                    combined_data["human_text"] = human_item.get('solution_text', 'NA')
                    combined_data["question_number"] = human_item.get('question_number', solution_number)
                    print(f"Human OCR solution_text: {repr(combined_data['human_text'])}")
                else:
                    print(f"Warning: {human_solution_path} is not a list or is empty.")
            except Exception as e:
                print(f"Error reading {human_solution_path}: {e}")
    else:
        print(f"File not found: {human_solution_path}")
    
    # Gemini OCR
    if os.path.exists(gemini_solution_path):
        with open(gemini_solution_path, 'r') as gemini_file:
            try:
                gemini_data = json.load(gemini_file)
                if isinstance(gemini_data, list) and len(gemini_data) > 0:
                    gemini_item = gemini_data[0]
                    combined_data["gemini_text"] = gemini_item.get('solution_text', 'NA')
                    print(f"Gemini OCR solution_text: {repr(combined_data['gemini_text'])}")
                else:
                    print(f"Warning: {gemini_solution_path} is not a list or is empty.")
            except Exception as e:
                print(f"Error reading {gemini_solution_path}: {e}")
    else:
        print(f"File not found: {gemini_solution_path}")
    
    # Create output directory
    output_dir = f'/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/tables/{solution_folder}'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Save as JSON file
    json_filename = os.path.join(output_dir, f"{file_prefix}_solution_{solution_number}_table.json")
    with open(json_filename, 'w') as json_file:
        json.dump(combined_data, json_file, indent=2)
    print(f"Created {json_filename}")

def process_folders(human_ocr_base_path, gemini_ocr_base_path):
    human_ocr_path = os.path.join(human_ocr_base_path, 'chemistry/Chemistry_human/solution_chapters')
    gemini_ocr_path = os.path.join(gemini_ocr_base_path, 'chemistry/Chemistry_Gemini/solutions')
    
    for solution_folder in os.listdir(human_ocr_path):
        solution_folder_path_human = os.path.join(human_ocr_path, solution_folder)
        if not os.path.isdir(solution_folder_path_human):
            continue
        for fname in os.listdir(solution_folder_path_human):
            if fname.endswith('.json') and '_solution_' in fname:
                try:
                    file_prefix = fname.split('_solution_')[0]
                    solution_number = int(fname.split('_solution_')[1].split('.')[0])
                except Exception:
                    print(f"Filename parse error: {fname}")
                    continue
                compare_ocr_folders(human_ocr_base_path, gemini_ocr_base_path, solution_folder, solution_number, file_prefix)

# Set paths to your directories
human_ocr_base_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr'
gemini_ocr_base_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr'

# Process the folders
process_folders(human_ocr_base_path, gemini_ocr_base_path)

Processing: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/solution_chapters/02_10021039271083421101694954824/02_solution_1.json
Processing: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_Gemini/solutions/02_10021039271083421101694954824/02_solution_1.json
Human OCR solution_text: '1.\n(3) Pbs – Reductant ; H₂O₂ - oxidant'
Gemini OCR solution_text: '1. (3) PbS - Reductant ; H₂O₂ - Oxidant'
Created /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/tables/02_10021039271083421101694954824/02_solution_1_table.json
Processing: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/solution_chapters/02_10021039271083421101694954824/02_solution_6.json
Processing: /Users/simrannaik/De

## to make cer

In [12]:
import os
import json
from difflib import ndiff

def char_error_rate(s1, s2):
    """
    Calculate the character error rate (CER) between two strings.
    Returns 'na' if either string is 'na'.
    """
    if s1 == "na" or s2 == "na":
        return "na"
    diff = list(ndiff(s1, s2))
    insertions = sum(1 for d in diff if d[0] == '+')
    deletions = sum(1 for d in diff if d[0] == '-')
    ref_len = len(s1)
    if ref_len == 0:
        return 0 if len(s2) == 0 else 1
    cer = (insertions + deletions) / ref_len
    return cer

def highlight_differences(s1, s2):
    """
    Highlight the differences between two strings.
    Returns 'na' if either string is 'na'.
    """
    if s1 == "na" or s2 == "na":
        return "na"
    diff = list(ndiff(s1, s2))
    result = []
    for d in diff:
        if d[0] == ' ':
            result.append(d[2])
        elif d[0] == '-':
            result.append(f"[-{d[2]}-]")
        elif d[0] == '+':
            result.append(f"[+{d[2]}+]")
    return ''.join(result)

def compare_ocr_folders_with_cer(human_ocr_path, gemini_ocr_path, solution_folder, solution_number, file_prefix):
    human_solution_path = os.path.join(
        human_ocr_path, 'chemistry/Chemistry_human/solution_chapters', solution_folder,
        f"{file_prefix}_solution_{solution_number}.json"
    )
    gemini_solution_path = os.path.join(
        gemini_ocr_path, 'chemistry/Chemistry_Gemini/solutions', solution_folder,
        f"{file_prefix}_solution_{solution_number}.json"
    )
    print(f"Processing: {human_solution_path}")
    print(f"Processing: {gemini_solution_path}")
    
    # Initialize content variables
    human_content = "NA"
    gemini_content = "NA"
    cer = "NA"
    highlight_diff = "NA"
    
    # Check if the Human OCR solution file exists and extract the 'solution_text'
    if os.path.exists(human_solution_path):
        with open(human_solution_path, 'r') as human_file:
            try:
                human_data = json.load(human_file)
                if isinstance(human_data, list) and len(human_data) > 0:
                    human_content = human_data[0].get('solution_text', 'NA')
                    print(f"Human OCR solution_text: {repr(human_content)}")
            except Exception as e:
                print(f"Error reading {human_solution_path}: {e}")
    else:
        print(f"File not found: {human_solution_path}")
    
    # Check if the Gemini OCR solution file exists and extract the 'solution_text'
    if os.path.exists(gemini_solution_path):
        with open(gemini_solution_path, 'r') as gemini_file:
            try:
                gemini_data = json.load(gemini_file)
                if isinstance(gemini_data, list) and len(gemini_data) > 0:
                    gemini_content = gemini_data[0].get('solution_text', 'NA')
                    print(f"Gemini OCR solution_text: {repr(gemini_content)}")
            except Exception as e:
                print(f"Error reading {gemini_solution_path}: {e}")
    else:
        print(f"File not found: {gemini_solution_path}")
    
    # Replace line breaks with <br> in both OCR contents (even if it's "NA")
    human_content_formatted = human_content.replace("\n", "<br>") if human_content != "NA" else "NA"
    gemini_content_formatted = gemini_content.replace("\n", "<br>") if gemini_content != "NA" else "NA"
    
    # If either Human OCR or Gemini OCR is "NA", set CER and highlight differences to "NA"
    if human_content == "NA" or gemini_content == "NA":
        cer = "NA"
        highlight_diff = "NA"
    else:
        # Calculate CER and highlight differences using formatted content
        cer = char_error_rate(human_content_formatted, gemini_content_formatted)
        highlight_diff = highlight_differences(human_content_formatted, gemini_content_formatted)
    
    # Create JSON structure with all comparison data
    comparison_data = {
        "question_number": solution_number,
        "human_text": human_content_formatted,
        "gemini_text": gemini_content_formatted,
        "cer": cer,
        "highlight_difference": highlight_diff
    }
    
    # Create output directory
    output_dir = f'/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/tables_cer/{solution_folder}'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Save as JSON file
    json_filename = os.path.join(output_dir, f"{file_prefix}_solution_{solution_number}_table.json")
    with open(json_filename, 'w') as json_file:
        json.dump(comparison_data, json_file, indent=2)
    print(f"Created {json_filename}")

def process_folders_with_cer(human_ocr_base_path, gemini_ocr_base_path):
    human_ocr_path = os.path.join(human_ocr_base_path, 'chemistry/chemistry_human/solution_chapters')
    gemini_ocr_path = os.path.join(gemini_ocr_base_path, 'chemistry/chemistry_Gemini/solutions')
    
    for solution_folder in os.listdir(human_ocr_path):
        solution_folder_path_human = os.path.join(human_ocr_path, solution_folder)
        if not os.path.isdir(solution_folder_path_human):
            continue
        for fname in os.listdir(solution_folder_path_human):
            if fname.endswith('.json') and '_solution_' in fname:
                try:
                    file_prefix = fname.split('_solution_')[0]
                    solution_number = int(fname.split('_solution_')[1].split('.')[0])
                except Exception:
                    print(f"Filename parse error: {fname}")
                    continue
                compare_ocr_folders_with_cer(human_ocr_base_path, gemini_ocr_base_path, solution_folder, solution_number, file_prefix)

# Set paths to your directories
human_ocr_base_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr'
gemini_ocr_base_path = '/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr'

# Process the folders
process_folders_with_cer(human_ocr_base_path, gemini_ocr_base_path)

Processing: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/solution_chapters/02_10021039271083421101694954824/02_solution_1.json
Processing: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_Gemini/solutions/02_10021039271083421101694954824/02_solution_1.json
Human OCR solution_text: '1.\n(3) Pbs – Reductant ; H₂O₂ - oxidant'
Gemini OCR solution_text: '1. (3) PbS - Reductant ; H₂O₂ - Oxidant'
Created /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/tables_cer/02_10021039271083421101694954824/02_solution_1_table.json
Processing: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/Chemistry_human/solution_chapters/02_10021039271083421101694954824/02_solution_6.json
Processing: /Users/simrannai

# to send gemini to get analysis on the human vs predictions

In [13]:
import os
import json
import time
from datetime import datetime
from dotenv import load_dotenv
import google.generativeai as genai
import sys
from pathlib import Path

# Handle both Jupyter notebook and standalone script environments
try:
    # This works in standalone Python scripts
    script_dir = Path(__file__).parent
except NameError:
    # This works in Jupyter notebooks
    script_dir = Path.cwd()

# Add the path to access prompt_store.py using relative path
project_root = script_dir.parent.parent.parent
ocr_path = project_root / "ocr"
sys.path.append(str(ocr_path))

from prompt_store import v14

# === Load API Key ===
load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_GEMINI_API"))

# Set model
model_name = "gemini-2.5-pro"
model = genai.GenerativeModel(model_name)

class ProcessingTracker:
    def __init__(self):
        self.total_files = 0
        self.processed_files = 0
        self.successful_files = 0
        self.failed_files = 0
        self.json_files = 0
        self.text_files = 0
        self.errors = []
        self.start_time = None
        self.end_time = None
        self.total_json_read_time = 0
        self.gemini_processing_time = 0
        self.file_save_time = 0
    
    def start_processing(self):
        self.start_time = datetime.now()
        print(f"🚀 Started processing at {self.start_time.strftime('%Y-%m-%d %H:%M:%S')}")
        print("=" * 70)
    
    def end_processing(self):
        self.end_time = datetime.now()
        duration = self.end_time - self.start_time
        print("\n" + "=" * 70)
        print("📊 PROCESSING SUMMARY")
        print("=" * 70)
        print(f"Total JSON files found:       {self.total_files}")
        print(f"Successfully processed:       {self.successful_files}")
        print(f"Failed to process:            {self.failed_files}")
        print(f"Valid JSON outputs:           {self.json_files}")
        print(f"Text outputs (invalid JSON):  {self.text_files}")
        print(f"JSON reading time:            {self.total_json_read_time:.2f}s")
        print(f"Gemini processing time:       {self.gemini_processing_time:.2f}s")
        print(f"File saving time:             {self.file_save_time:.2f}s")
        print(f"Total processing time:        {duration}")
        print(f"Completed at:                 {self.end_time.strftime('%Y-%m-%d %H:%M:%S')}")
        
        if self.errors:
            print(f"\n❌ ERRORS ENCOUNTERED ({len(self.errors)}):")
            print("-" * 50)
            for i, error in enumerate(self.errors, 1):
                print(f"{i}. {error}")
        else:
            print(f"\n✅ No errors encountered!")
        print("=" * 70)
    
    def add_error(self, error_msg):
        self.errors.append(error_msg)
        self.failed_files += 1

def send_json_and_prompt(input_json_path, prompt, output_json_dir, tracker, file_index):
    try:
        json_filename = os.path.basename(input_json_path)
        print(f"\n📄 [{file_index}/{tracker.total_files}] Processing: {json_filename}")
        
        # Step 1: Read the JSON file
        print("   📖 Reading JSON file...")
        start_time = time.time()
        with open(input_json_path, 'r', encoding='utf-8') as f:
            json_content = json.load(f)
        end_time = time.time()
        read_time = end_time - start_time
        tracker.total_json_read_time += read_time
        print(f"   ✅ JSON read in {read_time:.3f}s")

        # Step 2: Compose the prompt with JSON content
        print("   🔄 Preparing prompt...")
        full_prompt = f"{prompt}\n\n<JSON Input>\n{json.dumps(json_content, indent=2)}"

        # Step 3: Generate content with the AI model
        print("   🤖 Processing with Gemini...")
        start_time = time.time()
        response = model.generate_content(
            full_prompt,
            generation_config={"temperature": 0.2},
        )
        generated_text = response.text
        end_time = time.time()
        gemini_time = end_time - start_time
        tracker.gemini_processing_time += gemini_time
        print(f"   ⏱️  Gemini processing time: {gemini_time:.2f} seconds")

        # Step 4: Clean the response
        print("   🧹 Cleaning response...")
        if generated_text.strip().startswith('```json'):
            generated_text = generated_text.strip().removeprefix('```json').removesuffix('```').strip()
        elif generated_text.strip().startswith('```'):
            generated_text = generated_text.strip().removeprefix('```').removesuffix('```').strip()

        # Step 5: Extract metadata for output path
        solution_folder_name = os.path.basename(os.path.dirname(input_json_path))
        prefix = solution_folder_name.split('_')[0]  # Extract prefix part (e.g., "12")
        base_name = os.path.splitext(os.path.basename(input_json_path))[0]
        
        # Extract the solution number correctly from filenames like '12_solution_1_table.json'
        try:
            solution_number = base_name.split('_')[2]  # Correctly extract the solution number part
        except IndexError:
            solution_number = "unknown"
            print(f"   ⚠️  Could not extract solution number from {base_name}")

        # Step 6: Create the output folder and file path
        output_solution_dir = os.path.join(output_json_dir, solution_folder_name)
        os.makedirs(output_solution_dir, exist_ok=True)
        output_json_path = os.path.join(output_solution_dir, f"{prefix}_solution_{solution_number}_analysis.json")

        # Step 7: Save the result
        print("   💾 Saving results...")
        start_time = time.time()
        
        try:
            # Try to parse the generated text as JSON
            parsed_json = json.loads(generated_text)
            # Save as properly formatted JSON
            with open(output_json_path, 'w', encoding='utf-8') as out_file:
                json.dump(parsed_json, out_file, indent=2, ensure_ascii=False)
            print(f"   ✅ Valid JSON saved: {output_json_path}")
            tracker.json_files += 1
            tracker.successful_files += 1
            
        except json.JSONDecodeError as json_error:
            # If response is not valid JSON, save as text wrapped in JSON
            output_data = {
                "analysis_result": generated_text,
                "input_file": os.path.basename(input_json_path),
                "status": "raw_text_response",
                "json_error": str(json_error)
            }
            with open(output_json_path, 'w', encoding='utf-8') as out_file:
                json.dump(output_data, out_file, indent=2, ensure_ascii=False)
            print(f"   ⚠️  Invalid JSON, saved as wrapped text: {output_json_path}")
            print(f"   📝 JSON Error: {str(json_error)[:100]}...")
            tracker.text_files += 1
            tracker.successful_files += 1
        
        end_time = time.time()
        save_time = end_time - start_time
        tracker.file_save_time += save_time
        
        tracker.processed_files += 1

    except FileNotFoundError:
        error_msg = f"File: {json_filename} - File not found"
        print(f"   ❌ File not found: {input_json_path}")
        tracker.add_error(error_msg)
    except json.JSONDecodeError as e:
        error_msg = f"File: {json_filename} - Invalid input JSON: {str(e)}"
        print(f"   ❌ Invalid input JSON: {str(e)}")
        tracker.add_error(error_msg)
    except Exception as e:
        error_msg = f"File: {json_filename} - Processing error: {str(e)}"
        print(f"   ❌ Error processing {json_filename}: {str(e)}")
        tracker.add_error(error_msg)

# Function to process all .json files in a directory
def process_all_json_files(input_dir, output_dir, prompt):
    tracker = ProcessingTracker()
    
    if not os.path.exists(input_dir):
        print(f"❌ Input directory does not exist: {input_dir}")
        return tracker
    
    # First, count all JSON files
    print(f"🔍 Scanning for JSON files in: {input_dir}")
    json_files = []
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".json"):
                json_files.append(os.path.join(root, file))
    
    tracker.total_files = len(json_files)
    print(f"📁 Found {tracker.total_files} JSON files")
    
    if tracker.total_files == 0:
        print("❌ No JSON files found in the specified directory")
        return tracker
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    print(f"📂 Output directory: {output_dir}")
    
    tracker.start_processing()
    
    # Process each JSON file
    for index, json_path in enumerate(json_files, 1):
        send_json_and_prompt(json_path, prompt, output_dir, tracker, index)
        
        # Show progress
        progress = (index / tracker.total_files) * 100
        print(f"   📈 Progress: {progress:.1f}% ({index}/{tracker.total_files})")
    
    tracker.end_processing()
    return tracker

# === Example Usage ===
if __name__ == "__main__":
    print("🎯 JSON OCR Analysis Processor with Gemini AI")
    print("=" * 70)
    
    # Input directory containing .json files (relative to script location)
    input_json_dir = script_dir / "tables"
    
    # Output directory (relative to script location)
    output_json_dir = script_dir / "table_analysis"
    
    # Convert to strings for compatibility
    input_json_dir = str(input_json_dir)
    output_json_dir = str(output_json_dir)
    
    print(f"📂 Input directory:  {input_json_dir}")
    print(f"📂 Output directory: {output_json_dir}")
    print(f"🤖 Using model:      {model_name}")
    print(f"📋 Using prompt:     v14 (OCR Quality Analysis)")
    
    # Start processing all JSON files in the input directory
    result_tracker = process_all_json_files(input_json_dir, output_json_dir, v14)
    
    # Final status
    if result_tracker.total_files > 0:
        success_rate = (result_tracker.successful_files / result_tracker.total_files) * 100
        print(f"\n🎉 Overall success rate: {success_rate:.1f}%")
        
        if result_tracker.failed_files > 0:
            print(f"⚠️  {result_tracker.failed_files} files failed to process")
        else:
            print("🎊 All files processed successfully!")
    else:
        print("❌ No files were processed")

# For Jupyter notebook usage
def run_json_analysis():
    """Function to call from Jupyter notebook"""
    print("🎯 JSON OCR Analysis Processor with Gemini AI")
    print("=" * 70)
    
    # Input directory containing .json files (relative to current working directory)
    input_json_dir = script_dir / "tables"
    
    # Output directory (relative to current working directory)
    output_json_dir = script_dir / "table_analysis"
    
    # Convert to strings for compatibility
    input_json_dir = str(input_json_dir)
    output_json_dir = str(output_json_dir)
    
    print(f"📂 Input directory:  {input_json_dir}")
    print(f"📂 Output directory: {output_json_dir}")
    print(f"🤖 Using model:      {model_name}")
    print(f"📋 Using prompt:     v14 (OCR Quality Analysis)")
    
    # Start processing all JSON files in the input directory
    result_tracker = process_all_json_files(input_json_dir, output_json_dir, v14)
    
    # Final status
    if result_tracker.total_files > 0:
        success_rate = (result_tracker.successful_files / result_tracker.total_files) * 100
        print(f"\n🎉 Overall success rate: {success_rate:.1f}%")
        
        if result_tracker.failed_files > 0:
            print(f"⚠️  {result_tracker.failed_files} files failed to process")
        else:
            print("🎊 All files processed successfully!")
    else:
        print("❌ No files were processed")
    
    return result_tracker

🎯 JSON OCR Analysis Processor with Gemini AI
📂 Input directory:  /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/tables
📂 Output directory: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/table_analysis
🤖 Using model:      gemini-2.5-pro
📋 Using prompt:     v14 (OCR Quality Analysis)
🔍 Scanning for JSON files in: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/tables
📁 Found 24 JSON files
📂 Output directory: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/table_analysis
🚀 Started processing at 2025-07-30 19:03:33

📄 [1/24] Processing: 02_solution_2_table.json
   📖 Reading JSON file...
   ✅ JSON read in 0.002s
   🔄 Preparing prompt...
   🤖 Processing with Gemini...
   ⏱️  Gemini processing time: 15.09 seconds
  

## merge the cer and gemini tables

In [14]:
import os
import json

base_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry"
analysis_dir = os.path.join(base_dir, "table_analysis")
cer_dir = os.path.join(base_dir, "tables_cer")  # assuming this has the JSON files with CER data
final_dir = os.path.join(base_dir, "final_tables")

def read_json_file(json_path):
    """Read and return JSON data from file"""
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except (json.JSONDecodeError, FileNotFoundError) as e:
        print(f"Error reading {json_path}: {e}")
        return None

def write_json_file(json_path, data):
    """Write JSON data to file"""
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def merge_json_data(analysis_data, cer_data):
    """Merge analysis data with CER data"""
    # Start with CER data as base
    merged_data = cer_data.copy() if cer_data else {}
    
    # Add analysis fields from analysis_data
    if analysis_data:
        # Add analysis-specific fields
        merged_data.update({
            "type_of_error": analysis_data.get("type_of_error", "N/A"),
            "discrepancy_analysis": analysis_data.get("discrepancy_analysis", "N/A"),
            "has_errors": analysis_data.get("has_errors", False)
        })
    else:
        # If no analysis data, set defaults
        merged_data.update({
            "type_of_error": "N/A",
            "discrepancy_analysis": "N/A", 
            "has_errors": False
        })
    
    return merged_data

# Loop over all subfolders in table_analysis
for folder_id in os.listdir(analysis_dir):
    analysis_path = os.path.join(analysis_dir, folder_id)
    cer_path = os.path.join(cer_dir, folder_id)
    out_path = os.path.join(final_dir, folder_id)
    
    if not os.path.isdir(analysis_path):
        continue
    
    # Check if corresponding cer folder exists, if not skip
    if not os.path.isdir(cer_path):
        print(f"Skipping {folder_id}: no matching folder in tables directory")
        continue
        
    os.makedirs(out_path, exist_ok=True)

    for fname in os.listdir(analysis_path):
        if not fname.endswith(".json"):
            continue
            
        analysis_file = os.path.join(analysis_path, fname)
        
        # Create corresponding CER file name (adjust naming as needed)
        # Assuming analysis files end with '_analysis.json' and cer files end with '_table.json'
        cer_fname = fname.replace("_analysis.json", "_table.json")
        cer_file = os.path.join(cer_path, cer_fname)
        
        if not os.path.exists(cer_file):
            print(f"Skipping {fname} in {folder_id}: no matching file {cer_fname} in tables")
            continue

        # Read both JSON files
        analysis_data = read_json_file(analysis_file)
        cer_data = read_json_file(cer_file)
        
        if analysis_data is None and cer_data is None:
            print(f"Skipping {fname} in {folder_id}: both files failed to load")
            continue

        # Merge the data
        merged_data = merge_json_data(analysis_data, cer_data)
        
        # Create output filename
        out_fname = fname.replace("analysis", "final").replace("solution", "final")
        out_file = os.path.join(out_path, out_fname)
        
        # Write merged JSON
        write_json_file(out_file, merged_data)
        print(f"Saved combined JSON: {out_file}")

print("JSON merging completed!")

Saved combined JSON: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/final_tables/02_10021039271083421101694954824/02_final_9_final.json
Saved combined JSON: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/final_tables/02_10021039271083421101694954824/02_final_13_final.json
Saved combined JSON: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/final_tables/02_10021039271083421101694954824/02_final_4_final.json
Saved combined JSON: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/final_tables/02_10021039271083421101694954824/02_final_3_final.json
Saved combined JSON: /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/final_tables/02_1002103927108342

## make a final table 

In [15]:
import os
import json

base_dir = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry"
final_tables_dir = os.path.join(base_dir, "final_tables")
output_file = os.path.join(base_dir, "final_table.json")

all_data = []

def read_json_file(json_path):
    """Read and return JSON data from file"""
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except (json.JSONDecodeError, FileNotFoundError) as e:
        print(f"Error reading {json_path}: {e}")
        return None

for folder_id in os.listdir(final_tables_dir):
    folder_path = os.path.join(final_tables_dir, folder_id)
    if not os.path.isdir(folder_path):
        continue
    
    for fname in os.listdir(folder_path):
        if not fname.endswith(".json"):
            continue
            
        file_path = os.path.join(folder_path, fname)
        json_data = read_json_file(file_path)
        
        if json_data is None:
            print(f"Warning: {file_path} could not be read.")
            continue
        
        # Add filename to the data for tracking
        json_data["file_name"] = fname
        json_data["folder_id"] = folder_id
        
        # Add to all_data list
        all_data.append(json_data)

# Write merged JSON
output_data = {
    "total_files": len(all_data),
    "source_directory": final_tables_dir,
    "merged_data": all_data
}

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print(f"Merged {len(all_data)} JSON files written to {output_file}")

Merged 24 JSON files written to /Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/final_table.json


## make a average cer and no of errors analysis

In [16]:
import os
import json

input_file = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/final_table.json"

cer_values = []
na_count = 0
row_count = 0

# Read JSON file
with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# Check if the expected structure exists
if "merged_data" not in data:
    raise Exception("No 'merged_data' found in JSON!")

merged_data = data["merged_data"]

# Process each JSON object in merged_data
for item in merged_data:
    row_count += 1
    
    # Check if 'cer' field exists
    if "cer" not in item:
        print(f"Warning: No 'cer' field in item {row_count}")
        continue
    
    cer_val = item["cer"]
    
    # Handle different types of values
    if cer_val is None or (isinstance(cer_val, str) and cer_val.lower() == "na"):
        na_count += 1
    else:
        try:
            # Convert to float if it's not already
            if isinstance(cer_val, str):
                cer_float = float(cer_val)
            else:
                cer_float = float(cer_val)
            cer_values.append(cer_float)
        except (ValueError, TypeError):
            print(f"Warning: Could not convert CER value '{cer_val}' to float in item {row_count}")
            na_count += 1

# Calculate average
average_cer = sum(cer_values) / len(cer_values) if cer_values else 0

print(f"Total data items: {row_count}")
print(f"Total 'na' in cer field: {na_count}")
print(f"Valid CER values: {len(cer_values)}")
print(f"Average cer (excluding 'na'): {average_cer}")

# Additional statistics
if cer_values:
    min_cer = min(cer_values)
    max_cer = max(cer_values)
    print(f"Minimum CER: {min_cer}")
    print(f"Maximum CER: {max_cer}")

Total data items: 24
Total 'na' in cer field: 1
Valid CER values: 23
Average cer (excluding 'na'): 0.4362783581826663
Minimum CER: 0.02654867256637168
Maximum CER: 1.256317689530686


In [20]:
# Copy this code into your Jupyter notebook cell - WITH DEBUGGING

import os
import json
from collections import Counter

input_file = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/final_table.json"

# List of error types to count
error_types = [
    "Spelling Difference",
    "Wording Differences", 
    "Capitalization Difference",  # This will combine all capitalization variants
    "Extra Content",
    "Punctuation", 
    "Numerical Discrepancies",
    "Missing Content",
    "Content Mix-up",
    "Omission"
]

# Define variations that should be grouped together
error_variations = {
    "Capitalization Difference": [
        "Capitalization Difference",
        "Capitilization Difference",  # Misspelled version
        "Capitalization"  # Just the word without "Difference"
    ]
}

counts = Counter()
omission_debug = []  # Track all omission-related entries

# Read JSON file
with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# Check if the expected structure exists
if "merged_data" not in data:
    raise Exception("No 'merged_data' found in JSON!")

merged_data = data["merged_data"]

# Find the correct field names
alternative_fields = {
    'type_of_error': ['type_of_error', 'error_type', 'Type of Error'],
    'gemini_text': ['gemini_text', 'gemini_ocr', 'Gemini_OCR'],
    'human_text': ['human_text', 'human_ocr', 'Human_OCR']
}

field_mapping = {}
if merged_data:
    sample_item = merged_data[0]
    for field_type, possible_names in alternative_fields.items():
        found_field = None
        for name in possible_names:
            if name in sample_item:
                found_field = name
                break
        if found_field:
            field_mapping[field_type] = found_field

print("=== Field Mapping ===")
for field_type, field_name in field_mapping.items():
    print(f"{field_type} -> {field_name}")

print(f"\n=== Error Type Analysis ===")
print(f"Total items to process: {len(merged_data)}")

# Process each JSON object in merged_data  
processed_count = 0
skipped_count = 0

def get_normalized_error_type(error_str):
    """
    Convert error variations to their standard form.
    """
    for standard_error, variations in error_variations.items():
        if error_str in variations:
            return standard_error
    return error_str

for i, item in enumerate(merged_data, 1):
    # Get values using the mapped field names
    if all(field_mapping.get(ft) in item for ft in ['type_of_error', 'gemini_text', 'human_text']):
        error_val = str(item[field_mapping['type_of_error']]) if item[field_mapping['type_of_error']] is not None else "N/A"
        gemini_ocr = str(item[field_mapping['gemini_text']]) if item[field_mapping['gemini_text']] is not None else "NA"
        human_ocr = str(item[field_mapping['human_text']]) if item[field_mapping['human_text']] is not None else "NA"
        
        # DEBUG: Check for omission mentions
        if "omission" in error_val.lower():
            omission_debug.append({
                'item_num': i,
                'error_val': error_val,
                'gemini_na': gemini_ocr.lower() == "na",
                'human_na': human_ocr.lower() == "na"
            })
        
        # FIXED LOGIC: Handle omission cases differently
        # For omission errors, it's expected that Gemini might be "NA" while Human has content
        is_omission_error = "omission" in error_val.lower()
        
        # Skip only if BOTH are NA, or if it's NOT an omission error and either is NA
        should_skip = False
        if gemini_ocr.lower() == "na" and human_ocr.lower() == "na":
            # Both are NA - always skip
            should_skip = True
        elif not is_omission_error and (gemini_ocr.lower() == "na" or human_ocr.lower() == "na"):
            # Not an omission error but one is NA - skip
            should_skip = True
        
        if should_skip:
            skipped_count += 1
            continue
        
        processed_count += 1
        
        # Handle comma-separated error types
        if error_val == "NO ERRORS":
            counts[error_val] += 1
        elif error_val not in ["N/A", "", "null"]:
            # Split by comma and check each individual error type
            individual_errors = [err.strip() for err in error_val.split(',')]
            
            found_known_error = False
            for individual_error in individual_errors:
                # Normalize the error type (combine variations)
                normalized_error = get_normalized_error_type(individual_error)
                
                if normalized_error in error_types:
                    counts[normalized_error] += 1
                    found_known_error = True
                elif individual_error == "NO ERRORS":
                    counts[individual_error] += 1 
                    found_known_error = True
            
            # If none of the individual errors were recognized, count as "Other"
            if not found_known_error:
                counts["Other"] += 1
                if i <= 10:  # Only print first 10 for brevity
                    print(f"Found unexpected error type(s): '{error_val}' in item {i}")
    else:
        skipped_count += 1

print(f"\nProcessed items: {processed_count}")
print(f"Skipped items: {skipped_count}")

# DEBUG: Show omission findings
print(f"\n=== OMISSION DEBUG ===")
print(f"Found {len(omission_debug)} items with 'omission' in error text:")
for debug_item in omission_debug[:10]:  # Show first 10
    print(f"  Item {debug_item['item_num']}: '{debug_item['error_val']}' (Gemini NA: {debug_item['gemini_na']}, Human NA: {debug_item['human_na']})")

# Print results - SHOW ALL ERROR TYPES INCLUDING ZEROS
print(f"\n=== Error Type Counts (Combined) ===")
for error_type in error_types + ["NO ERRORS"]:
    print(f"{error_type}: {counts[error_type]}")

if counts["Other"] > 0:
    print(f"Other/Unexpected: {counts['Other']}")

# Calculate percentages - SHOW ALL INCLUDING ZEROS
total_valid = sum(counts.values())
if total_valid > 0:
    print(f"\n=== Percentages (of {total_valid} valid items) ===")
    for error_type in error_types + ["NO ERRORS"]:
        percentage = (counts[error_type] / total_valid) * 100
        print(f"{error_type}: {percentage:.1f}%")

# Show top error types (only non-zero for this section)
print(f"\n=== Top Error Types (Combined) ===")
for error_type, count in counts.most_common():
    if count > 0:
        percentage = (count / total_valid) * 100
        print(f"{error_type}: {count} ({percentage:.1f}%)")

# Show what was combined
print(f"\n=== Combination Details ===")
for standard_error, variations in error_variations.items():
    print(f"{standard_error} includes: {', '.join(variations)}")

# ADDITIONAL DEBUG: Show processed omission items
print(f"\n=== PROCESSED OMISSION ITEMS ===")
omission_processed = [debug for debug in omission_debug if not (debug['gemini_na'] and debug['human_na'])]
print(f"Omission items that should be processed: {len(omission_processed)}")
for debug_item in omission_processed:
    print(f"  Item {debug_item['item_num']}: '{debug_item['error_val']}'")

=== Field Mapping ===
type_of_error -> type_of_error
gemini_text -> gemini_text
human_text -> human_text

=== Error Type Analysis ===
Total items to process: 24

Processed items: 24
Skipped items: 0

=== OMISSION DEBUG ===
Found 1 items with 'omission' in error text:
  Item 9: 'Omission' (Gemini NA: True, Human NA: False)

=== Error Type Counts (Combined) ===
Spelling Difference: 4
Wording Differences: 13
Capitalization Difference: 4
Extra Content: 10
Punctuation: 18
Numerical Discrepancies: 4
Missing Content: 13
Content Mix-up: 0
Omission: 1
NO ERRORS: 0

=== Percentages (of 67 valid items) ===
Spelling Difference: 6.0%
Wording Differences: 19.4%
Capitalization Difference: 6.0%
Extra Content: 14.9%
Punctuation: 26.9%
Numerical Discrepancies: 6.0%
Missing Content: 19.4%
Content Mix-up: 0.0%
Omission: 1.5%
NO ERRORS: 0.0%

=== Top Error Types (Combined) ===
Punctuation: 18 (26.9%)
Wording Differences: 13 (19.4%)
Missing Content: 13 (19.4%)
Extra Content: 10 (14.9%)
Capitalization Differ

In [18]:
import os
import json
from collections import defaultdict

input_file = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/final_table.json"

# Read JSON file
with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# Check if the expected structure exists
if "merged_data" not in data:
    raise Exception("No 'merged_data' found in JSON!")

merged_data = data["merged_data"]

# Look for folder_id field (with possible variations)
folder_id_fields = ['folder_id', 'folderId', 'FOLDER_ID', 'folder_name']
folder_id_field = None

if merged_data:
    sample_item = merged_data[0]
    for field in folder_id_fields:
        if field in sample_item:
            folder_id_field = field
            break

if folder_id_field is None:
    print("Available fields in the data:")
    if merged_data:
        for key in merged_data[0].keys():
            print(f"  - {key}")
    raise Exception("No folder_id field found in JSON data!")

print(f"Using field '{folder_id_field}' for folder IDs")
print(f"Total items to process: {len(merged_data)}")

# Initialize counter for all possible numbers (1-99 to be safe)
prefix_counts = defaultdict(int)

# Process each JSON object in merged_data
processed_count = 0
unmatched_folders = []

for i, item in enumerate(merged_data, 1):
    if folder_id_field not in item:
        print(f"Warning: Item {i} missing {folder_id_field} field")
        continue
    
    folder_id = item[folder_id_field]
    
    # Convert to string if needed
    folder_id = str(folder_id) if folder_id is not None else ""
    
    if not folder_id:
        print(f"Warning: Empty folder_id in item {i}")
        continue
    
    processed_count += 1
    
    # Extract numeric prefix (part before first underscore)
    if '_' in folder_id:
        prefix = folder_id.split('_')[0]
        
        # Check if prefix is numeric
        if prefix.isdigit():
            # Convert to int to remove leading zeros, then back to string
            numeric_prefix = str(int(prefix))
            prefix_counts[numeric_prefix] += 1
        else:
            unmatched_folders.append(folder_id)
    else:
        unmatched_folders.append(folder_id)

print(f"\nProcessed items: {processed_count}")
print(f"Items in merged_data: {len(merged_data)}")

# Find the range of numbers we actually have
if prefix_counts:
    max_num = max(int(k) for k in prefix_counts.keys())
    min_num = min(int(k) for k in prefix_counts.keys())
else:
    max_num = 15  # default range
    min_num = 1

# Extend range to show a reasonable range (1 to at least 15)
max_num = max(max_num, 15)

print(f"\n=== Folder ID Prefix Counts ===")
total_files = 0
zero_counts = []

for num in range(1, max_num + 1):
    count = prefix_counts[str(num)]
    total_files += count
    print(f"{num}_: {count}")
    
    if count == 0:
        zero_counts.append(str(num))

print(f"\nTotal files counted: {total_files}")

# Show prefixes with zero counts
if zero_counts:
    print(f"Prefixes with 0 files: {', '.join(zero_counts)}")

# Show unmatched folder IDs for debugging
if unmatched_folders:
    print(f"\n=== Unmatched Folder IDs ===")
    unique_unmatched = list(set(unmatched_folders))[:10]  # Show first 10 unique
    for folder in unique_unmatched:
        print(f"  {folder}")
    if len(unmatched_folders) > 10:
        print(f"  ... and {len(unmatched_folders) - 10} more")

# Show some sample folder IDs for debugging
print(f"\n=== Sample Folder IDs ===")
sample_count = min(5, len(merged_data))
for i in range(sample_count):
    if folder_id_field in merged_data[i]:
        folder_id = merged_data[i][folder_id_field]
        prefix = folder_id.split('_')[0] if '_' in str(folder_id) else "no_underscore"
        print(f"  {i+1}: {folder_id} -> prefix: {prefix}")

Using field 'folder_id' for folder IDs
Total items to process: 24

Processed items: 24
Items in merged_data: 24

=== Folder ID Prefix Counts ===
1_: 11
2_: 13
3_: 0
4_: 0
5_: 0
6_: 0
7_: 0
8_: 0
9_: 0
10_: 0
11_: 0
12_: 0
13_: 0
14_: 0
15_: 0

Total files counted: 24
Prefixes with 0 files: 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15

=== Sample Folder IDs ===
  1: 02_10021039271083421101694954824 -> prefix: 02
  2: 02_10021039271083421101694954824 -> prefix: 02
  3: 02_10021039271083421101694954824 -> prefix: 02
  4: 02_10021039271083421101694954824 -> prefix: 02
  5: 02_10021039271083421101694954824 -> prefix: 02


In [21]:
## to check the na , total questions

# Question Count and NA Pattern Analysis

import os
import json
from collections import Counter, defaultdict

input_file = "/Users/simrannaik/Desktop/solution_improvement/ds-prototypes/subjective_grading/solution_improvement/z2_ocr/chemistry/final_table.json"

print("🔍 QUESTION COUNT & NA PATTERN ANALYSIS")
print("=" * 60)

# Read JSON file
with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# Check if the expected structure exists
if "merged_data" not in data:
    raise Exception("No 'merged_data' found in JSON!")

merged_data = data["merged_data"]

# Find the correct field names
alternative_fields = {
    'question_number': ['question_number', 'Question_Number', 'questionNumber'],
    'gemini_text': ['gemini_text', 'gemini_ocr', 'Gemini_OCR'],
    'human_text': ['human_text', 'human_ocr', 'Human_OCR'],
    'folder_id': ['folder_id', 'folderId', 'FOLDER_ID', 'folder_name']
}

field_mapping = {}
if merged_data:
    sample_item = merged_data[0]
    for field_type, possible_names in alternative_fields.items():
        found_field = None
        for name in possible_names:
            if name in sample_item:
                found_field = name
                break
        if found_field:
            field_mapping[field_type] = found_field

print("=== Field Mapping ===")
for field_type, field_name in field_mapping.items():
    print(f"{field_type} -> {field_name}")

print(f"\n=== Basic Counts ===")
print(f"Total items in dataset: {len(merged_data)}")

# Initialize counters
question_numbers = set()
folder_question_pairs = set()
na_patterns = {
    'both_na': 0,           # Both Gemini and Human are NA
    'gemini_na_human_text': 0,  # Gemini NA, Human has text
    'human_na_gemini_text': 0,  # Human NA, Gemini has text
    'both_have_text': 0     # Both have text
}

folder_stats = defaultdict(lambda: {
    'total_questions': set(),
    'both_na': 0,
    'gemini_na_human_text': 0,
    'human_na_gemini_text': 0,
    'both_have_text': 0
})

# Process each item
for i, item in enumerate(merged_data, 1):
    # Extract required fields
    question_num = item.get(field_mapping.get('question_number'), 'unknown')
    gemini_text = str(item.get(field_mapping.get('gemini_text'), 'NA')).strip()
    human_text = str(item.get(field_mapping.get('human_text'), 'NA')).strip()
    folder_id = str(item.get(field_mapping.get('folder_id'), 'unknown'))
    
    # Add to question tracking
    question_numbers.add(question_num)
    folder_question_pairs.add((folder_id, question_num))
    
    # Track questions per folder
    folder_stats[folder_id]['total_questions'].add(question_num)
    
    # Determine NA pattern
    gemini_is_na = gemini_text.lower() in ['na', 'none', 'null', '']
    human_is_na = human_text.lower() in ['na', 'none', 'null', '']
    
    if gemini_is_na and human_is_na:
        na_patterns['both_na'] += 1
        folder_stats[folder_id]['both_na'] += 1
    elif gemini_is_na and not human_is_na:
        na_patterns['gemini_na_human_text'] += 1
        folder_stats[folder_id]['gemini_na_human_text'] += 1
    elif human_is_na and not gemini_is_na:
        na_patterns['human_na_gemini_text'] += 1
        folder_stats[folder_id]['human_na_gemini_text'] += 1
    else:
        na_patterns['both_have_text'] += 1
        folder_stats[folder_id]['both_have_text'] += 1

# Convert sets to counts for folder stats
for folder_id in folder_stats:
    folder_stats[folder_id]['total_questions'] = len(folder_stats[folder_id]['total_questions'])

print(f"\n=== Question Analysis ===")
print(f"Unique question numbers found: {len(question_numbers)}")
print(f"Unique folder-question pairs: {len(folder_question_pairs)}")

# Sort question numbers for display
try:
    sorted_questions = sorted([int(q) for q in question_numbers if str(q).isdigit()])
    print(f"Question number range: {min(sorted_questions)} to {max(sorted_questions)}")
    print(f"Question numbers: {sorted_questions}")
except:
    print(f"Question numbers (mixed types): {sorted(question_numbers)}")

print(f"\n=== NA Pattern Analysis ===")
total_items = len(merged_data)
for pattern, count in na_patterns.items():
    percentage = (count / total_items) * 100
    print(f"{pattern.replace('_', ' ').title()}: {count} ({percentage:.1f}%)")

print(f"\n=== Detailed NA Pattern Breakdown ===")
print(f"📊 Both Gemini & Human are NA:        {na_patterns['both_na']:3d} items")
print(f"🤖 Gemini NA, Human has text:         {na_patterns['gemini_na_human_text']:3d} items")
print(f"👤 Human NA, Gemini has text:         {na_patterns['human_na_gemini_text']:3d} items")
print(f"✅ Both have text:                    {na_patterns['both_have_text']:3d} items")
print(f"📝 Total items:                       {total_items:3d} items")

print(f"\n=== Folder-wise Statistics ===")
print(f"Total folders: {len(folder_stats)}")
print("\nTop 10 folders by question count:")
sorted_folders = sorted(folder_stats.items(), key=lambda x: x[1]['total_questions'], reverse=True)

for folder_id, stats in sorted_folders[:10]:
    print(f"\n📁 {folder_id}:")
    print(f"   Questions: {stats['total_questions']}")
    print(f"   Both NA: {stats['both_na']}, Gemini NA: {stats['gemini_na_human_text']}")
    print(f"   Human NA: {stats['human_na_gemini_text']}, Both text: {stats['both_have_text']}")

# Summary for research insights
print(f"\n=== Research Insights ===")
print(f"🔬 Data Coverage:")
print(f"   • Total questions analyzed: {len(folder_question_pairs)}")
print(f"   • Questions with some OCR data: {na_patterns['gemini_na_human_text'] + na_patterns['human_na_gemini_text'] + na_patterns['both_have_text']}")
print(f"   • Questions with complete comparison: {na_patterns['both_have_text']}")

print(f"\n🚫 Missing Data Patterns:")
if na_patterns['gemini_na_human_text'] > 0:
    print(f"   • Gemini failed to detect text that humans could read: {na_patterns['gemini_na_human_text']} cases")
if na_patterns['human_na_gemini_text'] > 0:
    print(f"   • Human transcription missing but Gemini detected text: {na_patterns['human_na_gemini_text']} cases")
if na_patterns['both_na'] > 0:
    print(f"   • Both sources missing (possible empty/unclear content): {na_patterns['both_na']} cases")

# Calculate success rates
if total_items > 0:
    gemini_detection_rate = ((na_patterns['both_have_text'] + na_patterns['human_na_gemini_text']) / total_items) * 100
    human_transcription_rate = ((na_patterns['both_have_text'] + na_patterns['gemini_na_human_text']) / total_items) * 100
    
    print(f"\n📈 Detection/Transcription Rates:")
    print(f"   • Gemini detection success rate: {gemini_detection_rate:.1f}%")
    print(f"   • Human transcription coverage: {human_transcription_rate:.1f}%")
    print(f"   • Both sources available: {(na_patterns['both_have_text']/total_items)*100:.1f}%")

print("=" * 60)

🔍 QUESTION COUNT & NA PATTERN ANALYSIS
=== Field Mapping ===
question_number -> question_number
gemini_text -> gemini_text
human_text -> human_text
folder_id -> folder_id

=== Basic Counts ===
Total items in dataset: 24

=== Question Analysis ===
Unique question numbers found: 13
Unique folder-question pairs: 24
Question number range: 1 to 13
Question numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]

=== NA Pattern Analysis ===
Both Na: 0 (0.0%)
Gemini Na Human Text: 1 (4.2%)
Human Na Gemini Text: 0 (0.0%)
Both Have Text: 23 (95.8%)

=== Detailed NA Pattern Breakdown ===
📊 Both Gemini & Human are NA:          0 items
🤖 Gemini NA, Human has text:           1 items
👤 Human NA, Gemini has text:           0 items
✅ Both have text:                     23 items
📝 Total items:                        24 items

=== Folder-wise Statistics ===
Total folders: 2

Top 10 folders by question count:

📁 02_10021039271083421101694954824:
   Questions: 13
   Both NA: 0, Gemini NA: 1
   Human NA: 0, B